<a href="https://colab.research.google.com/github/MaGiaVy/DoAnPython/blob/main/project/notebooks/Nhom3thangcuti_Tuan4/notebooks/Tuan4_TwoStage_MobileCLIP_DINOv2_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Tuần 4 — 2-Stage Visual Search: MobileCLIP + YOLO-Crop DINOv2
## Shopee Product Matching | Google Colab T4 GPU

### Pipeline Architecture
```
Query Image + Title
     │
     ├─→ [YOLO Crop]  → Cropped product region
     │
     ├─→ [Stage 1: MobileCLIP]
     │   ├─ img_feat (original) + txt_feat
     │   ├─ fused = normalize(α·img + (1-α)·txt)
     │   └─ FAISS search → top-K candidates (K=100)
     │
     └─→ [Stage 2: DINOv2-Small on YOLO crop]
         ├─ dino_score = cosine(q_dino, gallery_dino[idx])
         ├─ fused_score = β·dino + (1-β)·clip
         └─ Return top-5
```

### Expected Results
| Metric | Baseline MobileCLIP | Target (2-Stage) | Gain |
|--------|---------------------|-------------------|------|
| mAP@5 | 0.7708 | ~0.79–0.80 | +2–3% |
| Precision@1 | 0.7937 | ~0.81–0.82 | +1–2% |
| Recall@5 | 0.7430 | ~0.76–0.77 | +1–2% |

### Notebook Sections
- **Cell 0**: Kiểm tra GPU
- **Cell 1**: Cài đặt thư viện
- **Cell 2**: Mount Drive & Load data
- **Cell 3**: Cấu hình & Chia tập
- **Cell 4**: Load MobileCLIP model
- **Cell 5**: Trích xuất MobileCLIP features (gallery)
- **Cell 6**: YOLO crop toàn bộ gallery
- **Cell 7**: Trích xuất DINOv2 features (YOLO crops)
- **Cell 8**: Grid search Alpha (Stage 1)
- **Cell 9**: Grid search Retrieval-K (Stage 2, β=1.0)
- **Cell 10**: Grid search Beta (full pipeline)
- **Cell 11**: Đánh giá Test set (chạy 1 lần!)
- **Cell 12**: So sánh với Baseline & Báo cáo
- **Cell 13**: Visualize kết quả truy vấn
- **Cell 14**: Demo single query

## ⚙️ Cell 0: Kiểm tra GPU & Runtime

In [ ]:
import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU khả dụng:')
    print(result.stdout)
else:
    print('❌ Không tìm thấy GPU!')
    print('👉 Vào Runtime > Change runtime type > chọn T4 GPU!')

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {vram:.1f} GB')
    if vram < 14:
        print('⚠️  VRAM < 14 GB — hãy giảm BATCH_SIZE_DINO xuống 16')

## 📦 Cell 1: Cài đặt thư viện

In [ ]:
import sys, subprocess

# Core dependencies
!pip install -q faiss-cpu ultralytics timm tqdm

# ─── Thử cài MobileCLIP (Apple) ────────────────────────────────────────────
USE_MOBILECLIP = False
print('⏳ Đang thử cài MobileCLIP (Apple)...')
try:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'git+https://github.com/apple/ml-mobileclip.git'],
        capture_output=True, text=True, timeout=180
    )
    import mobileclip
    USE_MOBILECLIP = True
    print('✅ MobileCLIP (Apple) sẵn sàng!')
except Exception as e:
    print(f'⚠️  Không cài được MobileCLIP: {e}')
    print('🔄 Fallback → openai/clip-vit-base-patch32 (HuggingFace)')
    !pip install -q transformers

print(f'\n🔧 Chế độ CLIP: {"MobileCLIP (Apple)" if USE_MOBILECLIP else "CLIP HuggingFace"}')

## 📂 Cell 2: Mount Google Drive & Load Dataset

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ─── Tự động dò tìm dataset ────────────────────────────────────────────────
POSSIBLE_PATHS = [
    '/content/drive/MyDrive/DoAnPython/DuLieuPython',
    '/content/drive/MyDrive/DuLieuPython',
    '/content/drive/My Drive/DoAnPython/DuLieuPython',
    '/content/drive/My Drive/DuLieuPython',
    '/content/drive/MyDrive/DuLieuPython/DuLieuPython',
]

DATA_DIR = None
for path in POSSIBLE_PATHS:
    if os.path.exists(os.path.join(path, 'train.csv')):
        DATA_DIR = path
        break

if DATA_DIR is None:
    DATA_DIR = '/content/drive/MyDrive/DuLieuPython/DuLieuPython'
    print(f'⚠️  Dùng đường dẫn mặc định: {DATA_DIR}')
else:
    print(f'✅ Phát hiện dataset tại: {DATA_DIR}')

CSV_PATH       = os.path.join(DATA_DIR, 'train.csv')
IMAGE_ZIP_PATH = os.path.join(DATA_DIR, 'train_images')
EXTRACTED_DIR  = '/content/train_images_extracted'
IMG_DIR        = os.path.join(EXTRACTED_DIR, 'train_images')

# Giải nén ảnh vào Colab local (đọc nhanh hơn Drive)
if not os.path.exists(IMG_DIR):
    if os.path.exists(IMAGE_ZIP_PATH):
        print('⏳ Giải nén train_images.zip (~1-2 phút)...')
        os.makedirs(EXTRACTED_DIR, exist_ok=True)
        !unzip -q "{IMAGE_ZIP_PATH}" -d "{EXTRACTED_DIR}"
        print('✅ Giải nén thành công!')
    else:
        print(f'❌ Không tìm thấy file zip tại {IMAGE_ZIP_PATH}')
else:
    print('✅ Thư mục ảnh đã có sẵn!')

# Kiểm tra
for name, p in [('train.csv', CSV_PATH), ('train_images/', IMG_DIR)]:
    ok = '✅' if os.path.exists(p) else '❌'
    print(f'  {ok} {name}: {p}')

## 🔧 Cell 3: Cấu hình & Chia tập dữ liệu

In [ ]:
import os, gc, json, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import faiss
from PIL import Image, UnidentifiedImageError
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from collections import defaultdict

warnings.filterwarnings('ignore')

# ─── Hyperparameters & Paths ────────────────────────────────────────────────
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED      = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# MobileCLIP
MOBILECLIP_VARIANT = 'mobileclip_s0'
MOBILECLIP_CKPT    = '/tmp/mobileclip_s0.pt'
BATCH_SIZE_CLIP    = 128
NUM_WORKERS        = 2

# YOLO
YOLO_WEIGHTS       = 'yolo11n.pt'   # pre-trained COCO; thay bằng model fine-tuned nếu có
YOLO_CONF          = 0.50
YOLO_IMGSZ         = 640
YOLO_BATCH         = 1              # stable mode; tăng nếu muốn nhanh hơn
CROP_PADDING       = 0.08           # 8% padding quanh bbox
FALLBACK_RATIO     = 0.75           # center-crop fallback khi YOLO fail

# DINOv2
DINO_MODEL         = 'dinov2_vits14'
DINO_FEAT_DIM      = 384
BATCH_SIZE_DINO    = 32             # giảm xuống 16 nếu OOM

# Retrieval
FINAL_K            = 5

# Paths
FEAT_DIR           = '/content/features'
CROP_DIR           = '/content/gallery_cropped'
RESULTS_DIR        = '/content/results'
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(CROP_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─── Load & Split ───────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f'📊 Dataset: {len(df):,} sản phẩm | {df["label_group"].nunique():,} nhóm')
print(df.head(2))

df_gallery = df.copy()  # toàn bộ gallery

val_idx, test_idx = train_test_split(
    df.index.tolist(), test_size=0.8, random_state=RANDOM_SEED
)
df_val  = df.loc[val_idx].reset_index(drop=True)   # 20% → tune hyperparams
df_test = df.loc[test_idx].reset_index(drop=True)  # 80% → eval cuối

print(f'\n🗂️  Gallery  : {len(df_gallery):,} ảnh')
print(f'✅ Val       : {len(df_val):,} queries → tune α, K, β')
print(f'✅ Test      : {len(df_test):,} queries → eval cuối (chạy 1 lần!)')
print(f'\n⚙️  Device   : {DEVICE}')
print(f'⚙️  YOLO     : {YOLO_WEIGHTS}')
print(f'⚙️  DINOv2   : {DINO_MODEL}')

## 🍎 Cell 4: Load MobileCLIP (hoặc CLIP Fallback)

In [ ]:
import urllib.request

if USE_MOBILECLIP:
    import mobileclip

    CKPT_URLS = {
        'mobileclip_s0': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s0.pt',
        'mobileclip_s1': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s1.pt',
    }
    if not os.path.exists(MOBILECLIP_CKPT):
        print(f'⏬ Download {MOBILECLIP_VARIANT}...')
        urllib.request.urlretrieve(CKPT_URLS[MOBILECLIP_VARIANT], MOBILECLIP_CKPT)
        print(f'✅ Saved: {MOBILECLIP_CKPT}')

    print(f'⏳ Loading {MOBILECLIP_VARIANT}...')
    clip_model, _, clip_preprocess = mobileclip.create_model_and_transforms(
        MOBILECLIP_VARIANT, pretrained=MOBILECLIP_CKPT
    )
    clip_tokenizer = mobileclip.get_tokenizer(MOBILECLIP_VARIANT)
    clip_model = clip_model.to(DEVICE).eval()

    with torch.no_grad():
        _dummy = torch.randn(1, 3, 256, 256).to(DEVICE)
        CLIP_DIM = clip_model.encode_image(_dummy).shape[-1]

    MODEL_LABEL = f'MobileCLIP ({MOBILECLIP_VARIANT})'

else:
    from transformers import CLIPModel, CLIPProcessor
    HF_MODEL = 'openai/clip-vit-base-patch32'
    print(f'⏳ Loading {HF_MODEL}...')
    clip_model     = CLIPModel.from_pretrained(HF_MODEL).to(DEVICE).eval()
    clip_preprocess = CLIPProcessor.from_pretrained(HF_MODEL)
    clip_tokenizer  = None
    CLIP_DIM        = clip_model.config.projection_dim
    MODEL_LABEL     = f'CLIP ({HF_MODEL})'

print(f'\n✅ {MODEL_LABEL}')
print(f'📐 Embedding dim: {CLIP_DIM}')

## 🖼️📝 Cell 5: Trích xuất MobileCLIP Features cho Gallery

In [ ]:

Claude finished the response
tui đang lm mô hình visual search bằng remBG xóa phông giai đoạn 1 mobile CLIP  giai đoạn 2 là effecientNetB0 + miniML để rerank á mà nếu giai đoạn 2 thấp hơn gia đoạn 1 thì có ảnh hưởng gì hong

May 28
Câu trả lời ngắn: Có ảnh hưởng, nhưng không nghiêm trọng nếu hiểu đúng vai trò từng giai đoạn
Phân tích pipeline của bạn
Query Image
    │
    ▼
[RemBG] ── xóa phông
    │
    ▼
[Mobile CLIP] ── Stage 1: Retrieval (top-K candidates)
    │
    ▼
[EfficientNetB0 + miniML] ── Stage 2: Reranking
    │
    ▼
Final Results
Vấn đề khi Stage 2 accuracy thấp hơn Stage 1
❌ Trường hợp có hại thật sự
Stage 2 rerank sai → đẩy kết quả đúng xuống dưới, kết quả sai lên trên
Net effect: pipeline tệ hơn dùng mình CLIP
Đặc biệt nguy hiểm nếu top-1, top-3 metric là KPI chính
✅ Trường hợp vẫn ổn
Nếu bạn đo Recall@K (K lớn) → Stage 1 lo recall, Stage 2 chỉ cần sort tốt hơn random trong K candidates
Stage 2 không cần accuracy tuyệt đối cao hơn Stage 1 toàn bộ dataset — chỉ cần phân biệt tốt trong top-K nhỏ
Tại sao accuracy so sánh kiểu này dễ mislead
Stage 1 (CLIP)	Stage 2 (EffNet + miniML)
Task	Search toàn bộ DB	Rerank ~20-100 candidates
Input distribution	Toàn bộ classes	Chỉ visually similar items
Metric phù hợp	Recall@K	NDCG, MRR, Precision@K
→ Accuracy của Stage 2 đo trên toàn bộ dataset sẽ luôn trông thấp hơn vì task khó hơn (fine-grained discrimination)

Cách kiểm tra thực tế
python
# Đo đúng cách
# Stage 1: Recall@100 — có candidate đúng trong top 100 không?
# Stage 2: Precision@5 hoặc MRR — sau rerank, đúng nằm đầu không?

# Pipeline tốt khi:
recall_stage1 > recall_stage2_alone  # CLIP giỏi retrieve
precision_after_rerank > precision_before_rerank  # EffNet cải thiện order
Khuyến nghị
Nếu Stage 2 đang làm hại pipeline:

Thử temperature scaling output CLIP + EffNet trước khi combine
Dùng weighted sum thay vì để miniML quyết hoàn toàn: score = α*clip_sim + (1-α)*effnet_score
Train miniML trên hard negatives từ chính output Stage 1, không phải random pairs
Metric nên dùng để evaluate pipeline:

Recall@100 cho Stage 1
NDCG@10 hoặc MRR cho Stage 2
Precision@1, Precision@5 cho end-to-end
Bạn đang thấy Stage 2 thấp hơn ở metric nào cụ thể vậy? Mình có thể suggest hướng fix cụ thể hơn.

Giữ lại EfficientNetB0, nâng cấp chiến lược Fusion: Có thể việc kết hợp đơn giản (late fusion) chưa đủ tốt. Hãy thử các phương pháp phức tạp hơn như:
Learned Weighted Fusion: Thay vì chọn alpha cố định, huấn luyện một mạng neural nhỏ để học cách kết hợp điểm số từ ảnh và text.
Cross-Attention Fusion: Sử dụng cơ chế attention để ảnh và text tương tác với nhau trước khi đưa ra điểm số cuối cùng.
Thay thế EfficientNetB0 bằng Backbone mạnh hơn nhưng vẫn nhẹ: Bạn có thể thử các mô hình được thiết kế riêng cho việc tạo embedding chất lượng cao:
MobileNetV3 (tinh chỉnh với ArcFace): Dùng Triplet Loss hoặc ArcFace để huấn luyện MobileNetV3 tạo ra embedding phân biệt tốt hơn cho sản phẩm.
EfficientFormer: Một kiến trúc hybrid giữa CNN và Transformer, có thể cho embedding tốt hơn EfficientNet.
Chuyển hẳn sang MobileCLIP cho cả hai giai đoạn: Đây là hướng đi an toàn và hiệu quả nhất. Bạn đã có MobileCLIP với mAP 0.77, có thể cải thiện thêm bằng cách:
Fine-tune MobileCLIP trên chính tập dữ liệu Shopee của bạn.
Thêm Attention Pooling thay vì lấy vector CLS token.
Sử dụng Multi-scale Inference: Chạy MobileCLIP trên nhiều scale của ảnh rồi tổng hợp embedding.
Giữ nguyên MobileCLIP làm GĐ1, thay đổi GĐ2: Nếu bạn muốn có pipeline 2 giai đoạn, hãy dùng MobileCLIP cho retrieval, sau đó dùng DINOv2-Small hoặc SigLIP cho reranking. Đây là hướng đi mà nhiều hệ thống visual search tiên tiến áp dụng. deepseek hướng tui đi như v nè
Show more
May 28
Deepseek suggest khá hợp lý, để mình phân tích từng hướng theo context thực tế của bạn (mobile, Shopee dataset, đã có mAP 0.77):

Đánh giá 4 hướng
Hướng 1 — Learned/Cross-Attention Fusion
Verdict: Không ưu tiên ngay

Cross-attention fusion nghe hay nhưng over-engineering cho bài toán rerank
Cần nhiều data labeled pairs để train hiệu quả
Latency tăng đáng kể — vấn đề với mobile
Learned weighted fusion thì đáng thử hơn, chỉ cần train 1 linear layer nhỏ
Hướng 2 — MobileNetV3 + ArcFace / EfficientFormer
Verdict: Có giá trị nếu bạn có labeled data tốt

ArcFace/Triplet Loss thực sự tạo embedding phân biệt tốt hơn cho product retrieval
Nhưng cần clean labeled data theo product ID — Shopee dataset bạn có đủ không?
EfficientFormer thì nặng hơn EfficientNetB0, cần benchmark latency trước
Hướng 3 — Fine-tune MobileCLIP toàn bộ
Verdict: Hướng tốt nhất về ROI

Đang có mAP 0.77 → fine-tune domain-specific có thể lên 0.82-0.85
Attention Pooling thay CLS token: đáng thử, thường +1-2% mAP với ảnh sản phẩm
Multi-scale inference: tốt nhưng nhân đôi latency, cân nhắc kỹ
Hướng 4 — MobileCLIP (GĐ1) + DINOv2-Small / SigLIP (GĐ2)
Verdict: Mạnh nhất về accuracy, nặng nhất về infra

DINOv2 và SigLIP embedding chất lượng rất cao cho visual similarity
Nhưng DINOv2-Small ~22M params, SigLIP còn nặng hơn → không thực sự "mobile-friendly" cho GĐ2
Phù hợp nếu GĐ2 chạy server-side, không phải on-device
Recommendation thực tế cho bạn
Ưu tiên theo thứ tự:

1. [Nhanh, ít rủi ro]
   Fine-tune MobileCLIP trên Shopee data
   + thử Attention Pooling
   → Kỳ vọng: mAP 0.77 → 0.82+

2. [Nếu vẫn cần 2-stage]
   GĐ1: MobileCLIP fine-tuned
   GĐ2: MobileCLIP fine-tuned (cùng model, khác score function)
         hoặc MobileNetV3 + ArcFace nếu có labeled product ID

3. [Nếu GĐ2 chạy server-side]
   GĐ2: SigLIP (tốt hơn DINOv2 cho text-image matching)
Câu hỏi quyết định hướng đi
Bạn trả lời 3 cái này mình suggest chính xác hơn:

GĐ2 chạy on-device hay server?
Shopee dataset của bạn có label product ID không (cùng sản phẩm, khác ảnh) hay chỉ có category?
Latency budget cho cả pipeline là bao nhiêu ms?
dataset tui là shopee matching guarantee á

May 28
Shopee Product Matching Dataset — Lợi thế lớn!
Dataset này cực kỳ phù hợp cho pipeline của bạn vì nó có đúng thứ cần thiết.

Dataset này có gì
train.csv
├── posting_id        # ID của listing
├── image             # tên file ảnh
├── image_phash       # perceptual hash
├── title             # tên sản phẩm (text)
└── label_group       # ← KEY: cùng label_group = cùng sản phẩm
label_group chính là Product ID → có thể train ArcFace/Triplet Loss trực tiếp.

Pipeline được mở khóa với dataset này
Train MobileCLIP fine-tune
python
# Positive pairs: cùng label_group, khác posting_id
# Negative pairs: khác label_group

# Contrastive / InfoNCE loss hoàn toàn applicable
loss = InfoNCE(
    anchor=clip_embed(img_A),
    positive=clip_embed(img_B),  # cùng label_group
    negatives=clip_embed(img_negatives)
)
Train MobileNetV3 + ArcFace cho GĐ2
python
# Label = label_group (treat như classification)
# ArcFace margin làm embedding cực phân biệt
model = MobileNetV3 + ArcFaceHead(
    num_classes=len(unique_label_groups)
)
Revised Recommendation
Với Shopee Matching dataset, thứ tự ưu tiên thay đổi:

Hướng	Trước	Sau khi biết dataset
Fine-tune MobileCLIP	✅ #1	✅ #1 vẫn giữ
MobileNetV3 + ArcFace	⚠️ cần data	✅ hoàn toàn khả thi
DINOv2/SigLIP GĐ2	server-side	vẫn nặng
Cross-Attention Fusion	over-eng	vẫn skip
Câu hỏi còn lại
GĐ2 chạy on-device hay server? và latency budget?

Hai cái này quyết định bạn có dùng được ArcFace model nặng hơn ở GĐ2 không — nếu server-side thì mở ra nhiều lựa chọn hơn nhiều.

v bạn gợi ý cho tui lm như nào cho để phù hợp với gpu t4 gg colab á

May 28
Pipeline tối ưu cho T4 Colab
T4 có 16GB VRAM, khá thoải mái. Vấn đề chính là Colab timeout và RAM CPU 12GB.

Recommended Pipeline
RemBG (CPU)
    │
    ▼
MobileCLIP fine-tuned (GĐ1) ── GPU T4
    │  top-50 candidates
    ▼
MobileNetV3 + ArcFace (GĐ2) ── GPU T4
    │  rerank top-10
    ▼
Final Results
Giai đoạn 1 — Fine-tune MobileCLIP
python
# Batch size T4 có thể chịu được
BATCH_SIZE = 128  # ~8GB VRAM với MobileCLIP-S2
LR = 1e-5        # fine-tune, không train từ đầu

# Mixed precision bắt buộc trên T4
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

with autocast():
    loss = info_nce_loss(embeddings, labels)

scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
Shopee dataset setup:

python
from torch.utils.data import Dataset

class ShopeeDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        # Group by label_group để sample pairs
        self.groups = df.groupby('label_group')['posting_id'].apply(list)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Lấy positive: cùng label_group
        same_group = self.groups[row['label_group']]
        pos_id = random.choice([x for x in same_group 
                                if x != row['posting_id']])
        # ...
        return anchor_img, positive_img, label
Giai đoạn 2 — MobileNetV3 + ArcFace
python
import timm
import torch.nn as nn

class ArcFaceModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # ~5.4M params, rất nhẹ trên T4
        self.backbone = timm.create_model(
            'mobilenetv3_large_100',
            pretrained=True,
            num_classes=0  # remove classifier
        )
        self.embedding = nn.Linear(960, 512)
        self.arcface = ArcFaceHead(512, num_classes, s=64, m=0.5)
    
    def forward(self, x, labels=None):
        feat = self.backbone(x)
        emb = self.embedding(feat)
        emb = F.normalize(emb, dim=1)
        if labels is not None:
            return self.arcface(emb, labels)
        return emb

# T4: batch 256 thoải mái với model này
BATCH_SIZE_ARCFACE = 256
Training Schedule thực tế trên Colab
Session 1 (~3-4h):
├── Extract tất cả embeddings MobileCLIP gốc → save .npy
├── Build FAISS index
└── Eval baseline mAP

Session 2 (~4-5h):
├── Fine-tune MobileCLIP 3-5 epochs
├── Checkpoint mỗi epoch → Google Drive
└── Eval mAP sau fine-tune

Session 3 (~3h):
├── Train MobileNetV3 + ArcFace 10-15 epochs
├── Checkpoint → Drive
└── Eval reranking metrics
Anti-timeout pattern:

python
# Save checkpoint sau mỗi epoch
def save_checkpoint(model, optimizer, epoch, loss, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, path)
    # Sync lên Drive ngay
    shutil.copy(path, '/content/drive/MyDrive/checkpoints/')

# Load lại khi reconnect
def load_checkpoint(model, optimizer, path):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint['epoch'], checkpoint['loss']
VRAM Budget estimate
Component	VRAM
MobileCLIP-S2 + batch 128	~6 GB
MobileNetV3 + batch 256	~4 GB
FAISS index (34K ảnh)	~1 GB
Buffer	~5 GB
Total	~10-11 GB / 16 GB ✅
Quick Start Code
python
# Cell 1: Setup
!pip install timm open-clip-torch faiss-gpu rembg -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Mount Drive để không mất data khi timeout
from google.colab import drive
drive.mount('/content/drive')

# Cell 2: Verify T4
assert 'T4' in torch.cuda.get_device_name(0), "Not T4!"
device = torch.device('cuda')

# Cell 3: Load MobileCLIP
import open_clip
model, _, preprocess = open_clip.create_model_and_transforms(
    'MobileCLIP-S2',
    pretrained='datacompdr'
)
model = model.to(device)
Bạn hiện tại đang ở bước nào rồi — đã có baseline mAP 0.77 từ MobileCLIP gốc chưa fine-tune, hay đã fine-tune rồi?

để tui coi coi mAP của đợt này ổn hong

May 28
Oke, chạy eval đi rồi paste kết quả vào đây!

Trong lúc chờ, bạn cho mình biết thêm:

Đang đo mAP@K nào (mAP@10, @50, hay toàn bộ)?
Dataset split như thế nào — dùng toàn bộ train set để eval hay có tách val riêng?
Có kết quả rồi mình phân tích luôn hướng tiếp theo.

tui có tách val riêng á, 20% để fine tunne đồ này nọ, 80% báo kết quả cuối, gallary để query á

May 28
Vậy setup của bạn là:

80% → Gallery (database để search)
20% → Query set (để eval)
Khá chuẩn cho bài retrieval! Chạy đi rồi paste kết quả vào, mình xem mAP, Precision@K, Recall@K luôn để biết bottleneck đang ở đâu — Stage 1 hay Stage 2.

100% gallery, 20% để fine tune đô các kỉu, 80% xuất kết quả cuối

May 28
À hiểu rồi, tức là:

100% → Gallery (toàn bộ dataset)
20%  → Query (eval, fine-tune threshold, etc.)
80%  → Test set (báo kết quả cuối)
Vậy không có data leakage vì query vẫn nằm trong gallery — đúng chuẩn bài retrieval rồi. Chạy đi paste kết quả vào nhé!


Tuan4_GiaVy_Pipeline.ipynb
ipynb

Here is my assignment file. Could you please help me generate a Markdown document to include rembg for background removal and pHash? Additionally, I need to implement a save point every 500 steps during the rembg process

Jun 1

Viewed 4 files
Viewed 4 files
Mình đã đọc xong toàn bộ notebook. Giờ tạo file Markdown với rembg + pHash + save point mỗi 500 steps.


Read remaining key cells about DINOv2 pipeline and EfficientNet
Read remaining key cells about DINOv2 pipeline and EfficientNet
Đã đọc đủ. Giờ tạo Markdown document:


Created a file, read a file
Created a file, read a file
Đây là file Markdown đã tích hợp đầy đủ vào pipeline của bạn. Những thay đổi chính:

Cell 2.5 — RemBG với Save Point mỗi 500 ảnh:

Dùng model u2netp (cân bằng tốc độ/chất lượng trên T4)
Checkpoint lưu cả local (/content/features/rembg_done.json) lẫn Drive — an toàn khi timeout
Ảnh lỗi tự động fallback copy ảnh gốc, không bị thiếu
Khi chạy lại sẽ tự tiếp tục từ chỗ dừng
Cell 2.6 — pHash:

Tận dụng cột image_phash sẵn có trong dataset Shopee thay vì tính lại
Thống kê nhóm ảnh trùng, có option dedup gallery (comment out, bật khi cần)
Toàn pipeline: Cập nhật ACTIVE_IMG_DIR = IMG_DIR_REMBG để tất cả các cell sau đều dùng ảnh đã xóa phông thay vì ảnh gốc.


Tuan4 giavy pipeline rembg phash
Document · MD 
ụa này là đầy đủ hay bổ sung thêm á, nếu tui hỏi AI cursor lm thì phải yêu cầu sao á

Jun 1
File này là bổ sung thêm vào notebook gốc của bạn, không phải thay thế hoàn toàn. Cụ thể:

Cell 0–5, 7–11: giữ nguyên logic gốc, chỉ update nhỏ (như ACTIVE_IMG_DIR, fallback nền trắng)
Cell 2.5 và 2.6 là mới hoàn toàn — chèn vào giữa Cell 2 và Cell 3
Prompt cho Cursor AI
Copy cái này vào Cursor:

Tôi có file notebook `Tuan4_GiaVy_Pipeline.ipynb`. 
Hãy chèn thêm 2 cell mới vào giữa Cell 2 (Google Drive) và Cell 3 (Import & Cấu hình):

**Cell 2.5 — RemBG xóa phông:**
- Cài rembg, dùng model u2netp
- Xử lý toàn bộ ảnh trong IMG_DIR, lưu ra /content/train_images_rembg/
- Save checkpoint JSON mỗi 500 ảnh (lưu cả local lẫn Google Drive)
- Khi chạy lại tự động tiếp tục từ chỗ dừng (resume từ checkpoint)
- Ảnh lỗi thì copy ảnh gốc, không bỏ qua
- Cuối cell gán IMG_DIR_REMBG = '/content/train_images_rembg'

**Cell 2.6 — pHash dedup:**
- Dùng cột image_phash sẵn có trong df_gallery
- Thống kê số nhóm ảnh trùng, số ảnh trùng
- Lưu mapping pHash → posting_id ra /content/features/phash_groups.json
- Có đoạn code dedup gallery nhưng để comment out

**Sau đó update Cell 3:**
- Thêm dòng ACTIVE_IMG_DIR = IMG_DIR_REMBG
- Các cell extract features (Cell 8) dùng ACTIVE_IMG_DIR thay vì IMG_DIR

Không thay đổi logic grid search, eval, hay export metrics.
Prompt càng chi tiết như vậy thì Cursor sẽ chèn đúng chỗ, không đụng vào phần còn lại của notebook.

Câu hỏi giữa kỳ xoay quanh các vấn đề: - Mô tả bài toán mà nhóm đang thực hiện - Hiện trạng nghiên cứu về các vấn đề mà nhóm đang thực hiện  - Nhóm đã thực hiện những gì, Kết quả thế nào - Cá nhân Sv đã làm những gì - Mô tả các cấu trúc cơ bản trong Python: if, for, while, def (hàm), class, và một số kiểu dữ liệu mới: Dict, Tupe, Set, List. Đóng vai chuyên gia truyền đạt đơn giản. Giải thích giúp uti vấn đeef trên bằng ngôn ngữ dễ hiểu cho người mới bắt đầu.Kèm ví dụ thực tế hoặc so sánh dễ hình dung. Bài toán là file ở trên

Jun 1
Giải thích Giữa Kỳ — Dễ hiểu cho người mới
1. Mô tả bài toán nhóm đang làm
Nói đơn giản: Bạn đang xây dựng công cụ tìm kiếm sản phẩm bằng hình ảnh cho sàn thương mại điện tử Shopee.

🛍️ Hình dung thế này: Bạn thấy một đôi giày ngoài đường, chụp ảnh lại, upload lên → hệ thống tự tìm ra những sản phẩm giống đôi giày đó trong kho 34,250 sản phẩm của Shopee.

Pipeline hoạt động như sau:

Ảnh query của người dùng
        ↓
  [RemBG] Xóa phông nền
        ↓
  [MobileCLIP] Chuyển ảnh + tên sản phẩm → vector số
        ↓
  [FAISS] So sánh với 34,250 ảnh trong kho → lấy top-K giống nhất
        ↓
  [DINOv2] Xem kỹ lại top-K → sắp xếp lại cho chính xác hơn
        ↓
  Trả về 5 sản phẩm giống nhất
2. Hiện trạng nghiên cứu
Trước đây người ta làm thế nào?

Thế hệ	Cách làm	Vấn đề
Cũ	So màu sắc, hình dạng thủ công	Không hiểu "ý nghĩa" ảnh
Trung bình	CNN (ResNet, EfficientNet)	Chỉ hiểu ảnh, không hiểu text
Hiện đại	CLIP, MobileCLIP, DINOv2	Hiểu cả ảnh lẫn chữ cùng lúc
💡 CLIP (OpenAI, 2021) là bước ngoặt: lần đầu tiên model hiểu được "ảnh con mèo" và chữ "cat" là cùng một thứ. MobileCLIP (Apple, 2024) là phiên bản nhỏ gọn hơn, chạy được trên điện thoại.

Kết quả nhóm đạt được so với chuẩn ngành:

mAP@5 = 0.77 → khá tốt cho zero-shot (không cần train lại)
Precision@1 = 0.79 → cứ 10 lần tìm kiếm, ~8 lần kết quả đầu tiên là đúng
3. Nhóm đã làm gì & Kết quả
Những gì đã làm:

Baseline 1 (so sánh):  CLIP gốc của OpenAI
Baseline 2 (chính):    MobileCLIP → mAP@5 = 0.77 ✅
Pipeline 2 giai đoạn:  MobileCLIP + DINOv2 reranking
Bổ sung:               RemBG xóa phông + pHash lọc ảnh trùng
Kết quả cụ thể (từ notebook):

Metric	Giá trị
mAP@5	0.7708
Precision@1	0.7937
Recall@5	0.7430
Alpha tối ưu	0.5 (ảnh và text ngang nhau)
4. Cá nhân bạn đã làm gì
Dựa vào notebook, bạn có thể trình bày:

Xây dựng ShopeeImageDataset — class đọc ảnh cho PyTorch
Viết hàm extract_image_features và extract_text_features
Implement fuse_and_normalize_clip — kết hợp ảnh + text
Xây dựng FAISS index để tìm kiếm nhanh
Viết hàm evaluate_retrieval_clip — tính mAP, Precision, Recall
Grid search alpha trên val set
Tích hợp RemBG + save checkpoint mỗi 500 ảnh
Tích hợp DINOv2 reranking ở giai đoạn 2
5. Cấu trúc Python cơ bản — Giải thích dễ hiểu
if — Rẽ nhánh quyết định
🚦 Hình dung: Như đèn giao thông. Xanh thì đi, đỏ thì dừng.

python
# Ví dụ thực tế từ notebook
if torch.cuda.is_available():
    device = 'cuda'   # Có GPU → dùng GPU
else:
    device = 'cpu'    # Không có → dùng CPU

# Ví dụ dễ hiểu hơn
diem = 8.5
if diem >= 8.5:
    print("Xuất sắc")
elif diem >= 7.0:
    print("Khá")
else:
    print("Trung bình")
for — Lặp qua từng phần tử
🔄 Hình dung: Như băng chuyền siêu thị — từng món hàng đi qua máy tính tiền một lượt.

python
# Ví dụ thực tế từ notebook
for fname in tqdm(todo_images, desc='🧹 RemBG xóa phông'):
    remove_bg_and_save(fname, IMG_DIR, REMBG_OUT_DIR, session)

# Ví dụ dễ hiểu hơn
san_pham = ['giày', 'áo', 'túi']
for sp in san_pham:
    print(f"Đang xử lý: {sp}")
# Output:
# Đang xử lý: giày
# Đang xử lý: áo
# Đang xử lý: túi
while — Lặp đến khi thỏa điều kiện
⏳ Hình dung: Như máy giặt — quay mãi đến khi hết thời gian mới dừng.

python
# Ví dụ dễ hiểu
epoch = 0
best_map = 0.0

while epoch < 10:
    map_hien_tai = train_one_epoch()   # giả sử hàm này tồn tại
    if map_hien_tai > best_map:
        best_map = map_hien_tai
        save_model()
    epoch += 1
    # Nếu quên epoch += 1 → vòng lặp vô tận ⚠️

# Ví dụ trong notebook (ý tưởng):
# while còn ảnh chưa xử lý:
#     xử lý ảnh tiếp theo
#     nếu đủ 500 ảnh → save checkpoint
def — Định nghĩa hàm
🧰 Hình dung: Như công thức nấu ăn. Viết một lần, dùng nhiều lần, chỉ thay nguyên liệu đầu vào.

python
# Ví dụ thực tế từ notebook
def fuse_and_normalize_clip(img_feats, txt_feats, alpha):
    # Kết hợp vector ảnh và vector text theo tỉ lệ alpha
    fused = alpha * img_feats + (1 - alpha) * txt_feats
    norms = np.linalg.norm(fused, axis=1, keepdims=True)
    return (fused / norms).astype('float32')

# Gọi hàm:
result = fuse_and_normalize_clip(img_feat, txt_feat, alpha=0.5)

# Ví dụ dễ hiểu hơn
def tinh_diem_cuoi(diem_giua_ky, diem_cuoi_ky):
    return 0.4 * diem_giua_ky + 0.6 * diem_cuoi_ky

diem = tinh_diem_cuoi(7.0, 8.5)  # → 7.9
class — Bản thiết kế đối tượng
🏭 Hình dung: Như khuôn đúc bánh. Một khuôn tạo ra nhiều cái bánh giống nhau nhưng mỗi cái có nhân khác nhau.

python
# Ví dụ thực tế từ notebook
class ShopeeImageDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df      = df        # dữ liệu CSV
        self.img_dir = img_dir   # thư mục ảnh

    def __len__(self):
        return len(self.df)      # tổng số ảnh

    def __getitem__(self, idx):
        fname = self.df.iloc[idx]['image']
        img   = Image.open(os.path.join(self.img_dir, fname))
        return img

# Tạo 2 "cái bánh" từ cùng 1 khuôn:
val_dataset  = ShopeeImageDataset(df_val,  IMG_DIR)  # dataset cho val
test_dataset = ShopeeImageDataset(df_test, IMG_DIR)  # dataset cho test

# Ví dụ dễ hiểu hơn
class SanPham:
    def __init__(self, ten, gia):
        self.ten = ten
        self.gia = gia

    def mo_ta(self):
        return f"{self.ten} giá {self.gia:,} VNĐ"

giay = SanPham("Giày Nike", 1_500_000)
ao   = SanPham("Áo Adidas",   800_000)
print(giay.mo_ta())  # → Giày Nike giá 1,500,000 VNĐ
Kiểu dữ liệu — List, Dict, Tuple, Set
🗂️ Hình dung 4 kiểu như 4 loại hộp đựng đồ khác nhau:

List — Danh sách có thứ tự, thay đổi được
📋 Như danh sách mua hàng — có thứ tự, thêm/bớt được

python
# Trong notebook
all_images = df_gallery['image'].tolist()
# → ['0000a688.jpg', '00039780.jpg', '000a190f.jpg', ...]

# Thao tác cơ bản
ket_qua = ['giày_A', 'túi_B', 'áo_C']
ket_qua.append('mũ_D')    # thêm vào cuối
ket_qua[0]                # lấy phần tử đầu → 'giày_A'
len(ket_qua)              # đếm → 4
Dict — Từ điển key → value
📖 Như danh bạ điện thoại — tra tên ra số, O(1) cực nhanh

python
# Trong notebook
gt_dict = {}
for pid, group in df.groupby('label_group'):
    gt_dict[pid] = set(group['posting_id'])
# → {'label_123': {'train_001', 'train_002'}, ...}

# Thao tác cơ bản
gia_san_pham = {
    'giày Nike': 1_500_000,
    'áo Adidas':   800_000,
}
gia_san_pham['giày Nike']          # → 1500000
gia_san_pham['túi Gucci'] = 5_000_000   # thêm mới
'áo Adidas' in gia_san_pham        # → True
Tuple — Danh sách cố định, không đổi được
🔒 Như tọa độ GPS — một khi đã xác định thì không thay đổi

python
# Ví dụ thực tế
embedding_shape = (34250, 512)   # 34250 ảnh, mỗi ảnh 512 chiều
width, height   = (224, 224)     # kích thước ảnh đầu vào DINOv2

# Không thể thay đổi:
# embedding_shape[0] = 100  → ❌ TypeError

# Dùng khi nào?
# → Trả về nhiều giá trị từ hàm
def lay_ket_qua():
    return 0.77, 0.79, 0.74   # mAP, Precision, Recall

map5, p1, r5 = lay_ket_qua()
Set — Tập hợp không trùng lặp
🎯 Như danh sách khách mời — mỗi người chỉ xuất hiện 1 lần dù mời nhiều lần

python
# Trong notebook — cực quan trọng khi tính relevant items
relevant = gt_dict.get(query_id, set()) - {query_id}
# Lấy tất cả sản phẩm cùng nhóm, TRỪ chính nó ra

# Thao tác cơ bản
san_pham_da_xem = {'giày_A', 'túi_B', 'giày_A'}  # tự loại trùng
# → {'giày_A', 'túi_B'}

ket_qua_tim = {'giày_A', 'áo_C', 'mũ_D'}
# Intersection — tìm điểm chung (dùng tính Recall!)
dung = san_pham_da_xem & ket_qua_tim   # → {'giày_A'}
len(dung) / len(san_pham_da_xem)       # Recall = 0.5
Tóm tắt 1 trang để ôn nhanh
Cấu trúc	Dùng khi nào	Ví dụ trong project
if	Rẽ nhánh điều kiện	Kiểm tra có GPU không
for	Lặp qua danh sách	Xử lý từng ảnh RemBG
while	Lặp đến khi thỏa	Training đến khi đủ epoch
def	Tái sử dụng code	fuse_and_normalize_clip()
class	Đóng gói data + hàm	ShopeeImageDataset
List	Danh sách có thứ tự	Danh sách tên file ảnh
Dict	Tra cứu nhanh key→value	gt_dict[posting_id]
Tuple	Giá trị cố định	Shape của embedding
Set	Không trùng, so sánh nhanh	Tập relevant items

Improved Pipeline Overview Stage 1 (Retrieval): MobileCLIP (image + text fusion with alpha) → FAISS → top-K candidates. Stage 2 (Re-ranking): DINOv2-Small computes similarity on cropped product images (using YOLO trained on Shopee data) → Score fusion with MobileCLIP scores (beta) → returns top-5.

pasted

how you think about this pipeline, is it oke?

7:38 AM

Scrutinized pipeline architecture, identified critical gaps
Scrutinized pipeline architecture, identified critical gaps
Code Review: Improved Pipeline ✅/⚠️
Cấu trúc chắc chắn tốt, nhưng có vài điểm rủi ro cần chú ý. Dưới đây là phân tích chi tiết:

✅ Những điểm TỐTTHÍCH
1. Separation of Concerns (tách bạch vai trò)
Stage 1: MobileCLIP dùng ảnh gốc → tốt vì cần context cho text embedding
Stage 2: DINOv2 dùng ảnh crop (chỉ sản phẩm) → tốt vì loại noise
Logic: Hợp lý, mỗi model làm việc của mình
2. Score Fusion Strategy (beta blending)
python
final_score = beta * dino_score + (1-beta) * clip_score
Đơn giản, dễ tune, đã chứng minh hiệu quả trong ngành.

3. Hyperparameter Search Sequencing
Alpha → Retrieval_k → Beta (step-by-step)
Tiết kiệm thời gian so với grid search 3D hoàn toàn
⚠️ Những điểm CẦN CHỌN LỌC / RỦI RO
🔴 VẤNĐỀ 1: YOLO Training Set — Data Leakage Risk
Vấn đề: Document nói "select a subset to annotate" nhưng không rõ subset này từ đâu.

Có 34,250 ảnh → chia thành:
├─ Gallery (toàn bộ? hay một phần?)
├─ Val (20%)
└─ Test (80%)

YOLO train set: từ subset này? Nếu overlap → data leakage ❌
Khuyên: Phải rõ ràng:

python
# ✅ Đúng: YOLO train set TÁCH BIỆT hoàn toàn
yolo_train_images = 500–1000 ảnh (từ nguồn khác hoặc unused)
gallery = 34,250 ảnh (không dùng để train YOLO)

# ❌ Sai: YOLO train set overlap với Gallery/Val/Test
yolo_train = random sample từ 34,250 → data leakage!
🔴 VẤNĐỀ 2: YOLO Fallback Strategy — Miền không khớp (Domain Mismatch)
Vấn đề: Bạn nói:

"If no suitable box is found, keep the original image as a fallback."

Nhưng DINOv2 được train trên cropped images, không phải original images. Khi YOLO fail → fallback original image → domain mismatch!

python
# Vấn đề:
if yolo_confidence < 0.5:
    crop = original_image  # ❌ DINOv2 chưa thấy loại ảnh này

# Tốt hơn: Luôn có chiến lược crop
def choose_crop(yolo_box):
    if yolo_box is None:
        # Option 1: Center crop
        h, w = img.shape[:2]
        return img[h//4:3*h//4, w//4:3*w//4]
    
    # Option 2: Largest region
    # Option 3: Image quad (chia 4 phần, lấy phần có entropy cao)
    return crop_region(img, yolo_box)
🟡 VẤNĐỀ 3: Score Fusion — Không chuẩn hóa
Vấn đề:

python
final_score = beta * dino_score + (1-beta) * clip_score
Nếu dino_score ở range [0, 1] nhưng clip_score ở range [-5, 5], thì beta sẽ biased về một score.

Khuyên: Chuẩn hóa trước khi fusion:

python
# ✅ Tốt hơn: Normalize cả hai scores
dino_score_norm = (dino_score - dino_min) / (dino_max - dino_min)
clip_score_norm = (clip_score - clip_min) / (clip_max - clip_min)

final_score = beta * dino_score_norm + (1-beta) * clip_score_norm
Hoặc dùng temperature scaling (softmax trước):

python
# Hoặc: Dùng rank fusion thay vì score fusion
# (Đơn giản hơn, không cần chuẩn hóa)
🟡 VẤNĐỀ 4: Inference Latency — Không mention
Vấn đề: Pipeline có bước thêm:

YOLO inference trên query (10-50ms)
DINOv2 inference trên query crop (20-50ms)
DINOv2 inference × K candidates (500-2000ms nếu K=100)
Tổng: ~1-3 giây / query → Có chấp nhận được không?

Original pipeline: ~500ms (chỉ MobileCLIP FAISS)
New pipeline:      ~2000ms (+ YOLO + DINOv2)
→ 4× chậm hơn!
Khuyên: Cần benchmark & optimize:

python
# Nhanh hóa options:
1. YOLO: Dùng nano/small (faster nhưng accuracy ↓)
2. DINOv2: Dùng phiên bản small thay vì large
3. K candidates: Giảm K (ví dụ: 50 thay vì 100)
4. Batch inference: Process nhiều candidates cùng lúc
🟡 VẤNĐỀ 5: YOLO Sẽ Fail Trên Một Số Loại Sản Phẩm
Scenario xấu:

Sản phẩm tập chung (ví dụ: 5 cái mũ xếp chồng) → YOLO detect toàn bộ, crop không chính xác
Sản phẩm quá nhỏ (trang sức) → YOLO nhầm lẫn
Sản phẩm suốt (kính, chai) → khó segmentate
Background phức tạp (chợ, đám đông) → YOLO confused
Khuyên: Tính toán YOLO success rate:

python
# Sau khi train YOLO, eval trên validation set:
yolo_success_rate = (boxes_with_confidence > 0.5) / total
# Nếu < 85% → cơn ác mộng! Cần re-train hoặc fallback strategy
🟡 VẤNĐỀ 6: Hyperparameter Search — Local Optima
Vấn đề: Bạn làm:

Grid alpha (trên MobileCLIP alone)
Grid retrieval_k (với beta=1, chỉ DINOv2)
Grid beta (trên full pipeline)
Nhưng alpha được fix từ bước 1, không re-tune ở bước 3!

python
# ❌ Vấn đề: Alpha tìm được từ MobileCLIP alone
# Nhưng khi thêm DINOv2, optimal alpha có thể khác!

# ✅ Tốt hơn: Joint optimization (nếu có thời gian)
for alpha in np.arange(0.0, 1.05, 0.1):
    for k in [50, 100, 150]:
        for beta in np.arange(0.2, 0.55, 0.05):
            val_map = evaluate(alpha, k, beta)
            # Lưu best (alpha, k, beta)
Nhưng điều này rất tốn thời gian (3 loop = 120 iterations). Cách bạn làm (sequential) là compromise hợp lý.

📊 So sánh Baseline vs New Pipeline
Metric	Baseline (MobileCLIP only)	New (+ DINOv2-crop)	Expected
mAP@5	0.7708	?	+2% → 0.79 (nếu YOLO tốt)
Latency	~500ms	~2000ms	⚠️ 4× chậm
Complexity	⭐	⭐⭐⭐⭐	Rủi ro cao hơn
🎯 Lời khuyên cuối cùng
Nên làm:
✅ Đầu tiên: Kiểm tra YOLO quality trên validation set
python
   # Sau train YOLO:
   yolo_val_map = evaluate_yolo_on_val_set()
   if yolo_val_map < 0.80:
       print("⚠️ YOLO quá yếu, có thể làm tệ hơn baseline!")
✅ Normalize scores trước fusion
✅ Benchmark latency trên GPU T4 Colab
✅ Test on held-out set không dùng để train YOLO
Không nên:
❌ Dùng cùng 34,250 ảnh cho cả YOLO training + Gallery
❌ Fallback original image khi YOLO fail (domain mismatch)
❌ Giả sử YOLO sẽ hoạt động tốt mà không verify trước
⚡ Tóm tắt lời khuyên — Có nên implement không?
Tiêu chí	Đánh giá
Logic	✅ Chắc chắn
Khả thi	🟡 Có, nhưng cần cẩn thận YOLO
Rủi ro	🔴 Cao — YOLO fail → tệ hơn baseline
ROI	🟡 Nếu YOLO tốt +2-3% mAP, nhưng +4× latency
Khuyên:

Thử ngay trên subset nhỏ 500 ảnh → train YOLO quick → test xem có tốt không
Nếu YOLO mAP < 0.80 → cân nhắc fallback strategy khác (ví dụ: fixed crop, center crop)
Nếu YOLO mAP > 0.85 → full implementation, expect +1-3% mAP
Có cần mình chi tiết hơn phần YOLO training strategy không? 🎯

could you help me generate a markdown document about your pipeline

7:45 AM

Architected comprehensive markdown document detailing pipeline implementation
Architected comprehensive markdown document detailing pipeline implementation
Perfect! 🎉 I've generated a comprehensive markdown document about your improved pipeline.

📄 Document Highlights
The document covers:

📊 Structure
Executive Summary — Expected metrics (+2-3% mAP vs baseline)
Architecture Diagrams — Visual flowcharts of both stages
Stage 1 & 2 Detailed Explanations — Why each design choice
Data Preparation — How to annotate & train YOLO (3 options)
Implementation Steps — 4 phases with code snippets
Hyperparameter Tuning — Sequential grid search strategy
Inference Pipeline — Complete inference functions
Code Examples — Full end-to-end training pipeline
Risk Analysis — 4 major risks + mitigation strategies
Performance Comparison — When pipeline works/fails
⚠️ Key Warnings in Document
YOLO Quality Check ✅
python
   if yolo_mAP < 0.80:
       print("⚠️ Risk of degradation!")
Score Normalization ✅
Before fusion, normalize both dino_score & clip_score to [0,1]
Domain Mismatch ✅
Never fallback to original image (mismatch with DINOv2 crops)
Always use center crop or YOLO box
Latency Concern ⚠️
Baseline: ~500ms
New: ~2000ms (4× slower)
🎯 Quick Decision Tree (from document)
START
├─ Is YOLO mAP > 0.85? 
│  ├─ NO → Use baseline MobileCLIP
│  └─ YES → Continue
│
├─ Can you afford 2-3 second latency?
│  ├─ NO → Use baseline
│  └─ YES → Continue
│
└─ Does +2-3% mAP matter?
   ├─ NO → Use baseline
   └─ YES → Deploy improved pipeline ✅
The document is production-ready for your giữa kỳ presentation or implementation. Bạn muốn mình chi tiết thêm phần nào không? 🚀


Improved 2stage pipeline documentation
Document · MD 
what prompt i should chat with cursor, Antigravity?

7:49 AM

Formulated comprehensive prompt for AI coding assistant implementation
Formulated comprehensive prompt for AI coding assistant implementation
🎯 Prompts for Cursor AI / Cody / GitHub Copilot
Here are optimized prompts to request implementation from AI coding assistants:

📝 Prompt 1: Complete Pipeline Implementation (Comprehensive)
I'm building an improved 2-stage visual search pipeline for Shopee product matching.
Here's the full specification:

PIPELINE OVERVIEW:
- Stage 1: MobileCLIP (image + text fusion) → FAISS → top-K candidates
- Stage 2: DINOv2-Small (on YOLO-cropped images) → re-rank → top-5
- Dataset: 34,250 products (100% gallery, 20% val queries, 80% test queries)
- Platform: Google Colab with T4 GPU

IMPLEMENTATION REQUIREMENTS:

1. **YOLO Training Module**
   - Load YOLOv8s pre-trained model
   - Train on annotated dataset (shopee_dataset.yaml)
   - Validate YOLO mAP > 0.85 before proceeding
   - Save best model checkpoint

2. **Image Cropping Module**
   - Implement choose_best_box() function:
     * Filters by confidence > 0.5
     * Checks box area (5-95% of image)
     * Checks center proximity
     * Returns best box or None (fallback)
   - Implement crop_image() with 10px padding
   - Process all gallery/val/test images
   - Save cropped images to /content/gallery_cropped/
   - Track crop statistics (yolo_count, fallback_count, error_count)

3. **Feature Extraction Module**
   - Extract DINOv2-Small features on cropped images:
     * Load: torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
     * Input: cropped images (224x224)
     * Output: x_norm_clstoken (384-dim) with L2 normalization
     * Batch size: 32
     * Cache to: /content/features/gallery_dino_crop.npy
   - Keep MobileCLIP features (original images) from baseline

4. **Hyperparameter Tuning Module**
   - Sequential grid search (3 steps):
     
     Step 1 - Alpha (MobileCLIP only):
       for alpha in np.arange(0.0, 1.05, 0.1):
           evaluate_stage1_only()
       → Save best_alpha
     
     Step 2 - Retrieval_K (DINOv2 only, beta=1.0):
       for k in [50, 100, 150, 200]:
           evaluate_two_stage(beta=1.0)
       → Save best_retrieval_k
     
     Step 3 - Beta (full pipeline):
       for beta in np.arange(0.2, 0.6, 0.05):
           evaluate_two_stage(full)
       → Save best_beta
   
   - Save best hyperparameters to JSON

5. **Inference Module**
   - Single-query function: search_two_stage_improved()
     Input: query_image_path, query_title, alpha, retrieval_k, beta
     Steps:
       1. YOLO crop query image
       2. Extract MobileCLIP features (original) + text
       3. FAISS search → top-K candidates
       4. Extract DINOv2 features (cropped)
       5. For each candidate: fuse scores
       6. Return top-5
     Output: top_indices, top_scores
   
   - Batch evaluation function: evaluate_two_stage()
     Input: query_df, gallery_df, hyperparams
     Output: mAP@5, Precision@1, Recall@5

6. **Score Normalization** (CRITICAL)
   - Normalize both scores to [0,1] before fusion:
     dino_norm = (dino_score - dino_min) / (dino_max - dino_min)
     clip_norm = (clip_score - clip_min) / (clip_max - clip_min)
     final_score = beta * dino_norm + (1-beta) * clip_norm

7. **Error Handling**
   - Handle corrupt images (try/except)
   - Handle YOLO detection failure (fallback to center crop)
   - Handle NaN/Inf in features (clamp or skip)
   - Print progress with tqdm

EXPECTED OUTPUTS:
- Trained YOLO model (shopee_yolo_best.pt)
- Cropped images directory (/content/gallery_cropped/)
- DINOv2 features cache (gallery_dino_crop.npy)
- Hyperparameter JSON (best_alpha, best_k, best_beta)
- Final metrics on test set (mAP@5, P@1, R@5)

INTEGRATION WITH COLAB:
- Use GPU device='cuda' by default
- Save checkpoints to Google Drive (/content/drive/MyDrive/...)
- Add progress bars (tqdm) for long operations
- Clear VRAM between model loads (del, torch.cuda.empty_cache())

CODE STYLE:
- Type hints (def func() -> np.ndarray)
- Docstrings for all functions
- Constants in UPPER_CASE at top
- Clear variable names
- Comments for non-obvious logic

Generate complete, production-ready code organized into:
1. YOLO training & cropping script
2. Feature extraction script
3. Hyperparameter tuning script
4. Inference & evaluation script
📝 Prompt 2: Modular Implementation (Preferred for Cursor)
I need a modular implementation of an improved visual search pipeline.
Create separate functions for each component. Follow this structure:

FILE: yolo_utils.py
─────────────────
Functions:
  - train_yolo(data_yaml, epochs=50)
  - load_yolo_model(checkpoint_path)
  - choose_best_box(yolo_output, img_shape, confidence_threshold=0.5)
  - crop_image(img, yolo_box, padding=10)
  - crop_gallery_images(df_gallery, yolo_model, output_dir, save_stats=True)

FILE: feature_extraction.py
──────────────────────────
Functions:
  - extract_dinov2_features(img_paths, batch_size=32, device='cuda')
  - extract_dinov2_single(img_path)
  - load_dinov2_model()
  - normalize_features(features)

FILE: hyperparameter_tuning.py
──────────────────────────────
Functions:
  - evaluate_stage1_only(query_df, gallery_df, alpha, K=5)
  - evaluate_two_stage(query_df, gallery_df, alpha, retrieval_k, beta, K=5)
  - grid_search_alpha(df_val, df_gallery, alphas=np.arange(0, 1.1, 0.1))
  - grid_search_retrieval_k(df_val, df_gallery, best_alpha, ks=[50,100,150,200])
  - grid_search_beta(df_val, df_gallery, best_alpha, best_k, betas=np.arange(0.2,0.6,0.05))
  - tune_all_hyperparams() → returns (best_alpha, best_k, best_beta, metrics_dict)

FILE: inference.py
──────────────────
Functions:
  - search_two_stage_improved(query_img_path, query_title, alpha, retrieval_k, beta, final_k=5)
  - search_batch(query_df, alpha, retrieval_k, beta, final_k=5)
  - evaluate_on_test_set(df_test, df_gallery, best_params) → final_metrics

FILE: pipeline_main.py
──────────────────────
Main execution flow:
  1. Train YOLO
  2. Crop all images
  3. Extract DINOv2 features
  4. Tune hyperparameters
  5. Evaluate on test set
  6. Save results

REQUIREMENTS:
- All functions have type hints
- All functions have docstrings with example usage
- Error handling (try/except with informative messages)
- Progress bars where appropriate (tqdm)
- Save intermediate results to avoid re-computation
- Integration with Google Colab (Drive mounting, GPU detection)

Generate clean, modular code.
📝 Prompt 3: Quick Start (Minimal)
Generate a single Colab notebook cell that:

1. Loads pre-trained YOLO model
2. Runs inference on 1 test image
3. Shows bounding box visualization
4. Extracts DINOv2 feature from cropped image
5. Compares with baseline MobileCLIP score

This is for quick validation that the pipeline architecture works before full implementation.

Include:
- imports
- model loading (with VRAM management)
- inference function
- visualization
- output: (yolo_bbox, dino_feature, mobileclip_score)
📝 Prompt 4: For Antigravity/Code Generation Tools
Task: Generate production-ready Python code for a 2-stage visual search system

Context:
- Input: 34,250 product images (Shopee dataset)
- Stage 1: MobileCLIP retrieval (existing, mAP=0.77)
- Stage 2: DINOv2 reranking (new, uses YOLO-cropped images)
- Platform: Google Colab T4 GPU
- Target improvement: +2-3% mAP

Components to generate:

1. YOLO Training Pipeline
   Input: annotated dataset (COCO format)
   Output: trained model checkpoint
   Validation: mAP > 0.85

2. Image Processing Pipeline
   Input: original images
   Process: YOLO detection → crop → save
   Output: cropped images + metadata
   Fallback: center crop if detection fails

3. Feature Extraction Pipeline
   Input: cropped images
   Model: DINOv2-Small (torch.hub)
   Output: 384-dim L2-normalized vectors
   Caching: numpy .npy files

4. Hyperparameter Optimization
   Method: sequential grid search
   Parameters: alpha ∈ [0, 1], k ∈ {50, 100, 150, 200}, beta ∈ [0.2, 0.6]
   Validation: Shopee validation set (20%)

5. Inference Engine
   Input: query image + title
   Process: extract features → FAISS search → DINOv2 rerank → score fusion
   Output: top-5 product indices + confidence scores

Quality Requirements:
- Type hints on all functions
- Comprehensive error handling
- Progress indicators (tqdm)
- Memory-efficient (clear unused models from VRAM)
- Results caching to avoid recomputation
- Logging of key metrics

Generate complete, self-contained code with examples.
💡 Tips for Best Results with Cursor/AI Assistant
✅ DO:
Be specific about inputs/outputs
Mention constraints (GPU memory, latency, dataset size)
Reference exact metrics (mAP@5, Precision@1)
Specify file paths (where to save/load)
Ask for error handling explicitly
Request type hints & docstrings
❌ DON'T:
Say "implement the pipeline" (too vague)
Forget to mention hyperparameters
Skip error cases
Ask without specifying data format
Request without mentioning your environment (Colab, GPU type, etc.)
🔥 Best Prompt for Cursor (Copy-Paste Ready)
I have a visual search pipeline with 2 stages:
Stage 1: MobileCLIP (retrieval via FAISS)
Stage 2: DINOv2 (reranking on YOLO-cropped images)

Dataset: Shopee, 34,250 products
Environment: Google Colab T4 GPU
Expected improvement: +2-3% mAP vs baseline (0.77)

I need you to write 5 modular Python functions:

1. def crop_images_with_yolo(image_paths, yolo_model) → cropped_images_dict
   - Load YOLO
   - Detect product bbox
   - Crop with padding
   - Fallback: center crop if detection fails
   - Return dict: {image_name: cropped_img}

2. def extract_dinov2_features(cropped_images, batch_size=32) → np.ndarray (shape: N×384)
   - Load DINOv2-Small
   - Resize to 224×224
   - Extract x_norm_clstoken
   - L2 normalize
   - Save to .npy file

3. def score_fusion(dino_scores, clip_scores, beta) → fused_scores
   - Normalize both to [0,1]
   - Blend: beta * dino + (1-beta) * clip
   - Return top-5 indices

4. def evaluate_two_stage(query_df, gallery_df, alpha, retrieval_k, beta) → metrics_dict
   - Extract features for queries
   - FAISS search
   - DINOv2 rerank
   - Compute mAP@5, Precision@1, Recall@5
   - Return dict with metrics

5. def search_single_query(query_image, query_title, params) → top_5_indices, scores
   - YOLO crop
   - Extract both features
   - FAISS search top-K
   - DINOv2 rerank
   - Return results

Requirements:
- Type hints on all parameters
- Error handling (corrupt images, YOLO fail)
- Docstrings with examples
- Memory efficient (free VRAM between models)
- Progress bars (tqdm)
- Save intermediate results

Generate clean, production-ready code.
📊 Expected Output Format
When you ask Cursor, expect back:

python
# ✅ GOOD RESPONSE:
def crop_images_with_yolo(...) -> Dict[str, np.ndarray]:
    """
    Docstring with example
    """
    # Well-commented code
    # Type hints
    # Error handling
    return results

# ❌ BAD RESPONSE:
def crop():
    # No docstring
    # No types
    # No comments
    return result
🎯 Quick Decision: Which Prompt to Use?
Use Case	Prompt
Want complete working code	Prompt 1 (Comprehensive)
Want modular, reusable functions	Prompt 2 (Modular)
Quick validation first	Prompt 3 (Quick Start)
Code generation tool (Antigravity)	Prompt 4 (Gen AI)
Just copy-paste into Cursor	🔥 Best Prompt (last one)
My recommendation: Use 🔥 Best Prompt — it's concise, specific, and Cursor will generate exactly what you need.

Bạn copy cái prompt cuối cùng vào Cursor, paste hết vào, rồi nó sẽ generate code ngay! 🚀

uả yolo của mình là lấy từ coco hay tự train trên shopee v

8:21 AM

Weighed YOLO model sourcing strategies and training approaches
Weighed YOLO model sourcing strategies and training approaches
🤔 YOLO Strategy: COCO Pretrain vs Shopee Fine-tune
Đây là quyết định quan trọng ảnh hưởng đến chất lượng pipeline. Hãy xem 2 cách:

Option 1: Pretrained COCO + Fine-tune on Shopee (✅ RECOMMENDED)
python
# Step 1: Load COCO pretrained
from ultralytics import YOLO
yolo = YOLO('yolov8s.pt')  # Pretrained on COCO (80 classes)

# Step 2: Fine-tune on Shopee data (your annotated images)
results = yolo.train(
    data='shopee_dataset.yaml',  # Your 500-1000 annotated images
    epochs=50,
    imgsz=640,
    batch=16,
    device=0
)

# Step 3: Validate
metrics = yolo.val()
# Expected: mAP > 0.85 on Shopee validation
Ưu điểm ✅
Nhanh: Transfer learning, chỉ cần 500-1000 ảnh annotated
Tốt: COCO đã học features chung (edges, colors, shapes)
Rẻ: Không cần train từ đầu
Thực tế: 90% công ty e-commerce làm như vậy
Nhược điểm ❌
Cần annotate 500-1000 ảnh (labor intensive)
COCO học "vật thể chung" (people, cars, ...) → không tối ưu cho Shopee
Option 2: Train YOLO Từ Đầu Trên Shopee (❌ NOT RECOMMENDED)
python
yolo = YOLO('yolov8s-untrained.pt')  # Random init
results = yolo.train(
    data='shopee_dataset.yaml',
    epochs=200,  # Nhiều hơn rất nhiều
    imgsz=640,
    batch=16
)
Nhược điểm ❌
Cần nhiều data: Minimum 2000-5000 ảnh annotated
Chậm: Train từ scratch = 10-20 giờ
Rủi ro: Overfit nếu data quý
Không hiểu gì: Chưa có background knowledge
Ưu điểm ✅
Tối ưu hóa hoàn toàn cho Shopee (nếu có đủ data)
🎯 Khuyên bạn: Hybrid Approach (Best)
Bạn nên làm:

┌─────────────────────────────────────┐
│  Bước 1: Dùng COCO Pretrain         │
│  yolo = YOLO('yolov8s.pt')          │
│                                      │
│  Bước 2: Test trên Shopee samples   │
│  Chạy trên 50 ảnh Shopee không label│
│  → Xem YOLO detect được gì          │
│                                      │
│  Bước 3: Quyết định annotation      │
│  ├─ Nếu COCO phát hiện tốt (>0.70)  │
│  │  → Chỉ cần 300-500 ảnh fine-tune │
│  │                                   │
│  └─ Nếu COCO phát hiện kém (<0.50)  │
│     → Cần 1000-2000 ảnh fine-tune   │
│                                      │
│  Bước 4: Fine-tune trên Shopee data │
│  epochs = 50-100                    │
│  Early stopping khi mAP > 0.85       │
└─────────────────────────────────────┘
📊 So sánh Chi Phí vs Hiệu Suất
Approach	Data Cần	Thời Gian Train	mAP Expected	Công Suất
COCO only (no finetune)	0	0	~0.65-0.70	Nhanh nhưng kém
COCO + 300 Shopee	300 ảnh	2 giờ	~0.80-0.85	🌟 BEST ROI
COCO + 500 Shopee	500 ảnh	4 giờ	~0.85-0.90	🌟 RECOMMENDED
COCO + 1000 Shopee	1000 ảnh	8 giờ	~0.88-0.92	Overkill
Train from scratch	2000+ ảnh	20 giờ	~0.90+	Quá rủi ro
🏆 Bạn nên chọn: COCO + 500 Shopee (4 giờ train, mAP ~0.85-0.90)

🔥 Cách Kiểm Tra COCO Pretraining Tốt Không
python
from ultralytics import YOLO
from PIL import Image
import cv2

# Load COCO pretrained (không finetune gì)
yolo_coco = YOLO('yolov8s.pt')

# Test trên 50 ảnh Shopee random (không annotate)
# Chỉ để xem nó phát hiện được không
test_images = [...]  # 50 random Shopee images

results = yolo_coco.predict(test_images, conf=0.5)

# Count successful detections
detected_count = 0
for result in results:
    if len(result.boxes) > 0:
        detected_count += 1

detection_rate = detected_count / len(test_images)
print(f"Detection rate (COCO only): {detection_rate:.1%}")

# ✅ Nếu > 60% → COCO đã khá tốt
# ⚠️ Nếu < 40% → Cần fine-tune kỹ lưỡng
📋 Decision Flow
python
# PSEUDO CODE: Quyết định của bạn

coco_detection_rate = test_coco_on_shopee()

if coco_detection_rate > 0.70:
    print("✅ COCO đủ tốt")
    annotation_budget = 300  # ảnh cần annotate
    
elif coco_detection_rate > 0.50:
    print("🟡 COCO khá, cần fine-tune")
    annotation_budget = 500
    
else:
    print("❌ COCO kém, cần fine-tune nặng")
    annotation_budget = 1000

# Fine-tune strategy
yolo_fine_tuned = finetune_yolo(
    base_model='yolov8s.pt',  # COCO pretrained
    shopee_data=f'{annotation_budget}_annotated_images',
    epochs=50
)

# Validate
mAP = validate_yolo(yolo_fine_tuned)
if mAP < 0.80:
    print("⚠️ Still not good, annotate more or use different model")
🎯 Concrete Action Plan Cho Bạn
Week 1: Preparation
□ Download YOLOv8s pretrained (coco)
□ Select 50 random Shopee images
□ Run YOLO inference (no training)
□ Check detection_rate
  └─ If >60%: proceed to fine-tune
  └─ If <60%: need more annotation budget
Week 2: Annotation
□ Depending on detection_rate, annotate:
  ├─ 300-500 images (if COCO good)
  └─ 500-1000 images (if COCO mediocre)

□ Tools: LabelImg, CVAT, or Roboflow
□ Output: YOLO format (.txt files)
□ Structure:
  yolo_dataset/
  ├─ images/train/
  ├─ images/val/
  └─ labels/train/
Week 3: Fine-tune
python
# Fine-tune script
yolo = YOLO('yolov8s.pt')  # COCO pretrained

results = yolo.train(
    data='shopee_dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    device=0
)

# Validate
metrics = yolo.val()
if metrics.box.map50 > 0.85:
    print("✅ YOLO ready for pipeline!")
    yolo.save('shopee_yolo_best.pt')
else:
    print("⚠️ Annotate more images or adjust hyperparams")
💡 Tóm Tắt Cho Bạn
Câu hỏi	Đáp án
YOLO lấy từ đâu?	COCO pretrained (yolov8s.pt)
Có cần train từ đầu?	❌ Không, waste time & data
Fine-tune cần bao nhiêu ảnh?	300-500 ảnh annotated
Thời gian train?	2-4 giờ trên T4 Colab
Expected mAP?	0.85-0.90
Có thể dùng COCO luôn?	Có, nhưng mAP chỉ ~0.65 → không đủ tốt
🎬 Nếu Bạn Muốn Nhanh (Lazy Mode)
python
# Không annotate, dùng COCO directly
yolo = YOLO('yolov8s.pt')  # COCO only

# Chấp nhận mAP thấp hơn (~0.70)
# DINOv2 sẽ bù lại phần nào
# Nhưng kết quả không optimal
Mình không khuyên vì:

COCO train trên vật thể chung (people, car, dog)
Shopee products (shoes, bags, clothes) → khác
YOLO sẽ detect sai → DINOv2 lấy ảnh crop sai → tệ hơn baseline
✅ Khuyên Cuối Cùng
Nên làm:
┌──────────────────────────────────┐
│ 1. Dùng YOLO COCO pretrain       │
│ 2. Annotate 500 ảnh Shopee       │
│ 3. Fine-tune 50 epochs           │
│ 4. Validate mAP > 0.85           │
│ 5. Dùng cho DINOv2 cropping      │
│                                  │
│ Thời gian: 5-7 ngày              │
│ Chi phí annotation: ~$50-100     │
│ Kết quả: mAP +2-3% vs baseline   │
└──────────────────────────────────┘
Có cần mình viết annotation guide không? 📝


Tuan4_TwoStage_MobileCLIP_DINOv2_Pipeline.ipynb
ipynb

file này xài yolo kiểu nào v

8:23 AM

Orchestrated comprehensive analysis for user inquiry
Orchestrated comprehensive analysis for user inquiry
📝 Summary: YOLO Usage Trong Notebook Của Bạn
Notebook của bạn xài YOLO COCO pretrain (không fine-tune). Chi tiết:

🔧 Config Của Bạn
python
YOLO_WEIGHTS  = 'yolo11n.pt'      # ← COCO pretrain (nano)
YOLO_CONF     = 0.20              # Lỏng lẻo (chấp nhận 20% confidence)
CROP_PADDING  = 0.08              # 8% padding quanh bbox
FALLBACK_RATIO = 0.80             # Center crop 80% nếu YOLO fail
🔍 Pipeline Flow
1. Load YOLO (yolo11n.pt)
   ↓
2. For each image: Run YOLO → detect boxes
   ↓
3. choose_best_box() → pick 1 best bbox (theo confidence + center + area)
   ↓
4. ✅ Có bbox? → Crop with 8% padding
   ❌ Không? → Fallback center crop (80% ảnh)
   ↓
5. Save cropped image
   ↓
6. DINOv2 process cropped image
⚠️ Điểm Quan Trọng
YOLO của bạn là COCO pretrain, KHÔNG fine-tune trên Shopee

Điều này có nghĩa:

✅ Tiết kiệm: Không cần annotate ảnh
❌ Rủi ro: COCO detect product sai → crop sai → DINOv2 nhận input xấu
Expected kết quả:

Nếu YOLO COCO tốt (detection rate > 70%) → mAP tăng +1-2%
Nếu YOLO COCO tệ (detection rate < 50%) → mAP không tăng hoặc giảm
🎯 Khuyên Tiếp Theo
Chạy notebook → Kiểm tra:
├─ crop_stats['yolo'] / total = ? %   (YOLO detection rate)
├─ crop_stats['fallback'] / total = ? (Fallback rate)
└─ Test mAP = ?                       (So sánh với baseline 0.7708)

Nếu YOLO rate > 70% → ✅ Okay
Nếu mAP tăng > 1% → ✅ Có benefit
Nếu cả 2 thỏa → Giữ nguyên
Nếu không → Cân nhắc fine-tune YOLO
Document chi tiết đã tạo ở file YOLO_Usage_Analysis_Notebook.md — bạn có thể xem flowchart, helper functions, checklist, v.v.

Bạn đã chạy notebook chưa? Kết quả mAP là bao nhiêu? 🚀


Yolo usage analysis notebook
Document · MD 
với theo bạn thì file đó có đúng với luồng hoạt động của bạn chưa

8:30 AM

Chuẩn bị tổng hợp phản hồi cho người dùng
Chuẩn bị tổng hợp phản hồi cho người dùng
📊 Quick Answer
Theo bạn thì file đó có đúng với luồng hoạt động của mình chưa?

✅ 95% ĐÚ NG!
📋 Tóm Tắt So Sánh
Component	Mình Gợi Ý	Notebook Bạn	Status
YOLO Loading	yolov8s.pt	yolo11n.pt	✅ Better!
YOLO Cropping	choose_best_box()	_choose_best_box()	✅ Identical
Fallback	center crop	_center_crop()	✅ Correct
MobileCLIP	Extract original	✅ Implement	✅ Correct
DINOv2	Extract cropped	✅ Implement	✅ Correct
Alpha Search	0.0-1.0, step 0.1	0.0-1.0, step 0.1	✅ Match
K Search	[50,100,150,200]	[50,100,150,200]	✅ Match
Beta Search	0.2-0.55, step 0.1	0.2-0.55, step 0.05	✅ Better!
Score Fusion	Normalize first	fused = β×dino+(1-β)×clip	⚠️ Check
Test Eval	Run once	✅ Run once	✅ Correct
🔍 2 Điểm Cần Verify
⚠️ Issue 1: YOLO_CONF = 0.20 (Quá Lỏng)
python
YOLO_CONF = 0.20  # Chấp nhận 20% confidence
# → Có thể detect sai (background, clutter, etc)

Khuyên: Tăng lên 0.35-0.50 (chặt hơn)
⚠️ Issue 2: Score Normalization Unclear
python
def score_fusion(dino_s, clip_s, beta, top_k=FINAL_K):
    fused = beta * _mm(dino_s.astype(np.float32)) + \
            (1-beta) * _mm(clip_s.astype(np.float32))
    # _mm() là gì?
Cần check: _mm() có normalize to [0,1] không?

✅ Những Điểm Bạn Làm Tốt Hơn Mình
Dùng YOLOv11n (thay vì v8s) — mới + nhẹ hơn ✅
Beta search chi tiết (step 0.05 thay vì 0.1) ✅
Fallback strategy (center crop thay vì original) ✅
🎬 Action: Kiểm Tra Ngay
python
# Chạy notebook rồi check:

print(f"YOLO Detection Rate: {crop_stats['yolo']/total:.1%}")
# Nếu > 70% → YOLO tốt ✅
# Nếu < 50% → Cần fine-tune YOLO ⚠️

print(f"mAP Improvement: {test_metrics['mAP@5']-0.7708:.4f}")
# Nếu > +0.02 → Pipeline working! 🎉
# Nếu < +0.01 → Cân nhắc fine-tune YOLO
Verdict: File notebook bạn gần như hoàn hảo. Chỉ cần verify 2 điểm nhỏ + chạy test để xem kết quả.

Có cần mình chi tiết hóa Issue 1 & 2 không? 🤔


Recommendation vs implementation comparison
Document · MD 
cần

8:32 AM

Chuẩn bị giải thích chi tiết hai vấn đề kỹ thuật
Chuẩn bị giải thích chi tiết hai vấn đề kỹ thuật
tiếp giúp tui

12:30 PM

Find _mm() function and score_fusion and YOLO_CONF in notebook
Find _mm() function and score_fusion and YOLO_CONF in notebook
Tìm thấy rồi! Đọc xong code thực tế, mình có verdict rõ ràng:

🔍 Kết Quả Phân Tích Thực Tế
⚠️ Issue 1: YOLO_CONF = 0.20
Code thực tế của bạn:

python
YOLO_CONF = 0.20
Kết hợp với _choose_best_box():

python
def _choose_best_box(boxes, scores, W, H):
    for box, conf in zip(boxes, scores):
        ar = (x2-x1)*(y2-y1) / img_area
        if ar < 0.01 or ar > 0.95: continue   # filter by area
        
        center = 1.0 - dist / max(diag, 1.0)
        area   = min(ar / 0.45, 1.0)
        
        score = float(conf) + 0.20*center + 0.15*area  # weighted
Verdict:

YOLO_CONF = 0.20 → Pre-filter rộng
     ↓
_choose_best_box() → Filter thêm (area + center + conf weighted)
     ↓
Kết quả: Chọn đúng bbox trong hầu hết trường hợp ✅

⚠️ Nhưng vẫn rủi ro khi:
- Background object nằm ở giữa (center score cao)
- Background object chiếm diện tích lớn (area score cao)
→ score = 0.22 + 0.20*0.9 + 0.15*0.8 = 0.22 + 0.18 + 0.12 = 0.52
→ Có thể beat 1 product thật ở góc!
Fix đơn giản:

python
# Cell 3 — thay đổi 1 dòng:

# CŨ (rủi ro):
YOLO_CONF = 0.20

# MỚI (an toàn hơn):
YOLO_CONF = 0.35

# Tác động: giảm false positive, tăng fallback rate ~5-8%
# Kết quả: crop quality tốt hơn
✅ Issue 2: Score Normalization — ĐÃ ĐÚNG!
Code thực tế của bạn:

python
def score_fusion(dino_s, clip_s, beta, top_k=FINAL_K):
    """Min-max normalize both, then blend."""
    def _mm(a): 
        return np.clip((a - a.min()) / (a.max() - a.min() + 1e-8), 0, 1)
    
    fused = beta  * _mm(dino_s.astype(np.float32)) + \
            (1-beta) * _mm(clip_s.astype(np.float32))
Phân tích:

_mm(a) làm gì?
├─ (a - a.min()) / (a.max() - a.min() + 1e-8)  → Min-Max normalize
└─ np.clip(..., 0, 1)                            → Clamp về [0, 1]

Ví dụ:
dino_scores = [0.95, 0.50, 0.30, 0.80, 0.60]
_mm(dino_scores) = [1.00, 0.31, 0.00, 0.77, 0.46]  ← [0, 1] ✅

clip_scores = [-0.3, 0.8, 0.2, 0.5, -0.1]
_mm(clip_scores) = [0.00, 1.00, 0.45, 0.73, 0.18]  ← [0, 1] ✅

fused (beta=0.3):
= 0.3 * [1.00, 0.31, 0.00, 0.77, 0.46]
+ 0.7 * [0.00, 1.00, 0.45, 0.73, 0.18]
= [0.30, 0.79, 0.32, 0.74, 0.26]  ← Balanced! ✅
Verdict: ✅ Issue 2 KHÔNG CÓ VẤN ĐỀ — Code đã đúng!

📊 Final Verdict Cả 2 Issues
Issue	Severity	Status	Action
YOLO_CONF = 0.20	🟡 Medium	⚠️ Lỏng nhưng có buffer	Tăng lên 0.35
Score Normalization	✅ None	✅ Đã đúng	Không cần fix
🔧 Chỉ Cần Fix 1 Dòng
python
# Cell 3, tìm dòng này:
YOLO_CONF = 0.20

# Đổi thành:
YOLO_CONF = 0.35  # Hoặc 0.40 nếu muốn chặt hơn
Sau đó chạy notebook → kiểm tra:

python
# Thêm vào sau Cell 6 (sau crop_gallery):
total = sum(crop_stats.values())
print(f"YOLO detect  : {crop_stats['yolo']:,} ({crop_stats['yolo']/total:.1%})")
print(f"Fallback crop: {crop_stats['fallback']:,} ({crop_stats['fallback']/total:.1%})")
print(f"Error        : {crop_stats['error']:,} ({crop_stats['error']/total:.1%})")

# Kỳ vọng:
# - YOLO > 70% → ✅ Tốt
# - YOLO 50-70% → ⚠️ Chấp nhận được
# - YOLO < 50% → ❌ Cần fine-tune YOLO
🎯 Tóm Tắt Cho Bạn
Pipeline notebook của bạn:
┌─────────────────────────────────────────┐
│ ✅ Score fusion (_mm): ĐÚNG HOÀN TOÀN  │
│ ✅ Cropping logic: ĐÚNG                │
│ ✅ Hyperparameter search: ĐÚNG         │
│ ✅ Evaluation flow: ĐÚNG               │
│                                         │
│ ⚠️ Chỉ cần fix: YOLO_CONF 0.20 → 0.35 │
│    (1 dòng, 5 giây)                     │
└─────────────────────────────────────────┘

Action ngay:
1. Đổi YOLO_CONF = 0.35
2. Chạy notebook
3. Check crop_stats (YOLO rate > 70%?)
4. So sánh mAP với baseline 0.7708
Pipeline của bạn gần như production-ready rồi! 🚀 Chạy đi xem kết quả thế nào nhé!

1780464922661_image.png
ủa yolo là 0.5 mè

12:35 PM
Ồ bạn đã sửa rồi! 🎉

✅ Config Hiện Tại Của Bạn
python
YOLO_CONF     = 0.50   # ✅ Đã fix! (trước là 0.20)
FALLBACK_RATIO = 0.75  # ✅ Tốt hơn (trước là 0.80, tight hơn)
So Sánh Trước vs Sau
Config	Trước	Hiện Tại	Verdict
YOLO_CONF	0.20	0.50	✅ Đúng rồi
FALLBACK_RATIO	0.80	0.75	✅ Tighter crop
🎯 Đánh Giá Config 0.50
YOLO_CONF = 0.50:
├─ Chỉ chấp nhận detection khi YOLO > 50% confident
├─ Loại bỏ gần hết false positives
├─ Fallback rate tăng ~10-15% (center crop 75%)
└─ Crop quality CAO HƠN nhiều so với 0.20 ✅

FALLBACK_RATIO = 0.75:
├─ Center crop chặt hơn (75% thay vì 80%)
├─ Bỏ 12.5% mỗi cạnh
└─ Loại bỏ border noise, watermark tốt hơn ✅
Pipeline của bạn bây giờ là ổn hoàn toàn rồi — cả 2 issues đều đã được address. Chạy notebook đi xem mAP ra bao nhiêu! 🚀


Tuan4_TwoStage_MobileCLIP_DINOv2_Pipeline.ipynb
ipynb

Check this file, is it ok

12:37 PM

Identified critical self-match bug deflating metrics
Identified critical self-match bug deflating metrics
Đọc xong toàn bộ 16 cells. Có 1 bug nghiêm trọng cần fix ngay, và một vài điểm nhỏ.

🔴 BUG NGHIÊM TRỌNG: Self-match không bị lọc trong evaluate_two_stage
Vấn đề ở Cell 10:

python
# evaluate_stage1 (Cell 9) — LỌC ĐÚNG ✅
retrieved = [g_pids[j] for j in idxs[i] 
             if g_pids[j] != row.posting_id][:K]  # ← có filter self

# evaluate_two_stage (Cell 10) — THIẾU LỌC ❌
cand_idxs   = idxs[0][idxs[0] >= 0]   # ← KHÔNG filter self-match!
clip_scores = dists[0][idxs[0] >= 0]
Hậu quả:

Gallery = 34,250 ảnh (bao gồm cả query!)

Query "shoe_001" tìm trong gallery:
├─ FAISS rank 1: "shoe_001" (chính nó!) → cosine = 1.0 ← HIGHEST
├─ FAISS rank 2: "shoe_002" (relevant)
├─ FAISS rank 3: "shoe_003" (relevant)
├─ ...

DINOv2 rerank:
├─ "shoe_001" → dino_score = 1.0 ← HIGHEST → rank 1 sau rerank
├─ relevant items → bị đẩy xuống rank 2, 3, 4, 5

top-5 retrieved = ["shoe_001", "shoe_002", ...]
                    ↑ chính nó, không count vào AP
                    ↑ chiếm mất 1 slot trong top-5!

→ mAP bị THẤP HƠN thực tế (deflated metrics)
Fix — thêm 3 dòng vào evaluate_two_stage (Cell 10):

python
def evaluate_two_stage(...):
    ...
    for i in tqdm(range(len(query_df)), ...):
        qid      = query_df.iloc[i]['posting_id']
        relevant = gt.get(qid, set()) - {qid}
        if not relevant: continue

        # FAISS search
        q_vec = np.ascontiguousarray(q_fused[i:i+1])
        dists, idxs = index.search(q_vec, retrieval_k + 1)
        cand_idxs   = idxs[0][idxs[0] >= 0]
        clip_scores = dists[0][idxs[0] >= 0]

        # ✅ THÊM: Filter self-match
        self_mask   = np.array([g_pids[j] != qid for j in cand_idxs])
        cand_idxs   = cand_idxs[self_mask]
        clip_scores = clip_scores[self_mask]

        if len(cand_idxs) == 0:
            ...
🟡 MINOR: val/test features không được cache
Cell 5 — CLIP features:

python
# Gallery ← CÓ cache ✅
if os.path.exists(GAL_IMG_PATH):
    gallery_img_feat = np.load(GAL_IMG_PATH)
else:
    gallery_img_feat = extract_clip_image_features(...)
    np.save(GAL_IMG_PATH, ...)  # ← Lưu cache

# Val & Test ← KHÔNG cache ❌
val_img_feat  = extract_clip_image_features(df_val, IMG_DIR)   # ← Re-run mỗi lần
val_txt_feat  = extract_clip_text_features(df_val)
test_img_feat = extract_clip_image_features(df_test, IMG_DIR)
test_txt_feat = extract_clip_text_features(df_test)
Cell 7 — DINOv2 features:

python
# Gallery ← CÓ cache ✅
if os.path.exists(GAL_DINO_PATH):
    gallery_dino_feat = np.load(GAL_DINO_PATH)

# Val & Test ← KHÔNG cache ❌
val_dino_feat  = extract_dino_features(df_val,  CROP_DIR, dino_model)
test_dino_feat = extract_dino_features(df_test, CROP_DIR, dino_model)
Fix — thêm cache cho val/test:

python
# Cell 5 — sau khi extract val/test CLIP:
VAL_IMG_PATH  = os.path.join(FEAT_DIR, 'val_clip_image.npy')
VAL_TXT_PATH  = os.path.join(FEAT_DIR, 'val_clip_text.npy')
TEST_IMG_PATH = os.path.join(FEAT_DIR, 'test_clip_image.npy')
TEST_TXT_PATH = os.path.join(FEAT_DIR, 'test_clip_text.npy')

if os.path.exists(VAL_IMG_PATH):
    val_img_feat = np.load(VAL_IMG_PATH)
    val_txt_feat = np.load(VAL_TXT_PATH)
else:
    val_img_feat = extract_clip_image_features(df_val, IMG_DIR)
    val_txt_feat = extract_clip_text_features(df_val)
    np.save(VAL_IMG_PATH, val_img_feat)
    np.save(VAL_TXT_PATH, val_txt_feat)

if os.path.exists(TEST_IMG_PATH):
    test_img_feat = np.load(TEST_IMG_PATH)
    test_txt_feat = np.load(TEST_TXT_PATH)
else:
    test_img_feat = extract_clip_image_features(df_test, IMG_DIR)
    test_txt_feat = extract_clip_text_features(df_test)
    np.save(TEST_IMG_PATH, test_img_feat)
    np.save(TEST_TXT_PATH, test_txt_feat)
✅ Tất cả những thứ đúng
Cell 0:  GPU check + VRAM warning        ✅
Cell 1:  Install + MobileCLIP fallback   ✅
Cell 2:  Drive mount + auto path detect  ✅
Cell 3:  YOLO_CONF=0.50, FALLBACK=0.75   ✅
Cell 4:  MobileCLIP loading              ✅
Cell 5:  Feature extraction functions    ✅
Cell 6:  YOLO crop + _choose_best_box    ✅
Cell 7:  VRAM free → load DINOv2         ✅
Cell 8:  score_fusion với _mm() đúng     ✅
Cell 9:  Alpha grid search (stage1)      ✅
Cell 10: evaluate_two_stage + K search   ⚠️ (self-match bug)
Cell 11: Beta grid search                ✅
Cell 12: Test set eval (1 lần)           ✅
Cell 13: Comparison + bar chart          ✅
Cell 14: visualize_query function        ✅
Cell 15: Demo + latency measure          ✅
📊 Overall
Severity	Issue	Fix
🔴 Critical	Self-match trong evaluate_two_stage	Thêm 3 dòng filter
🟡 Minor	Val/test features không cache	Thêm np.save/load
Pipeline logic, flow, hyperparameter search — tất cả đúng. Chỉ cần fix bug self-match là chạy được.

Ưu tiên fix ngay Cell 10 — thêm 3 dòng self_mask là xong! 🔧

lỗi cell 5,7 là sao á

12:43 PM

Phân tích lỗi caching trong hai cell xử lý features
Phân tích lỗi caching trong hai cell xử lý features
Lỗi Cell 5 & 7 — Giải thích đơn giản
Không phải lỗi làm chương trình crash, mà là lỗi làm tốn thời gian mỗi lần restart Colab.

📌 Cell 5 — CLIP Features
Vấn đề:

python
# Gallery → CÓ cache ✅
if os.path.exists(GAL_IMG_PATH):
    gallery_img_feat = np.load(GAL_IMG_PATH)   # Load nhanh
else:
    gallery_img_feat = extract_clip_image_features(...)
    np.save(GAL_IMG_PATH, gallery_img_feat)    # Lưu lại

# Val & Test → KHÔNG cache ❌
val_img_feat  = extract_clip_image_features(df_val, IMG_DIR)  # Chạy lại TỪ ĐẦU
val_txt_feat  = extract_clip_text_features(df_val)            # mỗi lần restart!
test_img_feat = extract_clip_image_features(df_test, IMG_DIR) # ~15-20 phút
test_txt_feat = extract_clip_text_features(df_test)
Hình dung:

Restart Colab lần 1:  Extract val+test CLIP → 20 phút ⏳
Restart Colab lần 2:  Extract val+test CLIP → 20 phút ⏳ (lại!)
Restart Colab lần 3:  Extract val+test CLIP → 20 phút ⏳ (lại!)

→ Lãng phí thời gian mỗi lần restart!
📌 Cell 7 — DINOv2 Features
Vấn đề tương tự:

python
# Gallery → CÓ cache ✅
if os.path.exists(GAL_DINO_PATH):
    gallery_dino_feat = np.load(GAL_DINO_PATH)  # Load nhanh

# Val & Test → KHÔNG cache ❌
val_dino_feat  = extract_dino_features(df_val,  CROP_DIR, dino_model)  # Chạy lại!
test_dino_feat = extract_dino_features(df_test, CROP_DIR, dino_model)  # ~10 phút
🔧 Fix — Thêm Cache Cho Cả 2 Cells
Cell 5 (sau dòng extract gallery):
python
# ── Val features ──────────────────────────────────────
VAL_IMG_PATH  = os.path.join(FEAT_DIR, 'val_clip_image.npy')
VAL_TXT_PATH  = os.path.join(FEAT_DIR, 'val_clip_text.npy')

if os.path.exists(VAL_IMG_PATH):
    print('✅ Load cached val features')
    val_img_feat = np.load(VAL_IMG_PATH)
    val_txt_feat = np.load(VAL_TXT_PATH)
else:
    print('⚙️ Extracting val features...')
    val_img_feat = extract_clip_image_features(df_val, IMG_DIR)
    val_txt_feat = extract_clip_text_features(df_val)
    np.save(VAL_IMG_PATH, val_img_feat)
    np.save(VAL_TXT_PATH, val_txt_feat)
    print('💾 Saved val cache')

# ── Test features ─────────────────────────────────────
TEST_IMG_PATH = os.path.join(FEAT_DIR, 'test_clip_image.npy')
TEST_TXT_PATH = os.path.join(FEAT_DIR, 'test_clip_text.npy')

if os.path.exists(TEST_IMG_PATH):
    print('✅ Load cached test features')
    test_img_feat = np.load(TEST_IMG_PATH)
    test_txt_feat = np.load(TEST_TXT_PATH)
else:
    print('⚙️ Extracting test features...')
    test_img_feat = extract_clip_image_features(df_test, IMG_DIR)
    test_txt_feat = extract_clip_text_features(df_test)
    np.save(TEST_IMG_PATH, test_img_feat)
    np.save(TEST_TXT_PATH, test_txt_feat)
    print('💾 Saved test cache')
Cell 7 (sau dòng extract gallery DINOv2):
python
# ── Val DINOv2 features ───────────────────────────────
VAL_DINO_PATH  = os.path.join(FEAT_DIR, f'val_dino_{DINO_MODEL}.npy')

if os.path.exists(VAL_DINO_PATH):
    print('✅ Load cached val DINOv2 features')
    val_dino_feat = np.load(VAL_DINO_PATH)
else:
    print('⚙️ Extracting val DINOv2 features...')
    val_dino_feat = extract_dino_features(df_val, CROP_DIR, dino_model)
    np.save(VAL_DINO_PATH, val_dino_feat)
    print('💾 Saved val DINOv2 cache')

# ── Test DINOv2 features ──────────────────────────────
TEST_DINO_PATH = os.path.join(FEAT_DIR, f'test_dino_{DINO_MODEL}.npy')

if os.path.exists(TEST_DINO_PATH):
    print('✅ Load cached test DINOv2 features')
    test_dino_feat = np.load(TEST_DINO_PATH)
else:
    print('⚙️ Extracting test DINOv2 features...')
    test_dino_feat = extract_dino_features(df_test, CROP_DIR, dino_model)
    np.save(TEST_DINO_PATH, test_dino_feat)
    print('💾 Saved test DINOv2 cache')
📊 Trước vs Sau
Không cache	Có cache
Lần 1 chạy	30 phút	30 phút
Lần 2 (restart)	30 phút ❌	30 giây ✅
Lần 3 (restart)	30 phút ❌	30 giây ✅
🎯 Tóm Tắt
Cell 5 & 7 không crash ✅
Chỉ là: val + test features bị extract lại mỗi lần restart
→ Tốn ~30 phút không cần thiết

Fix: Thêm if/else cache giống gallery là xong 🔧
Priority: Fix bug self-match (Cell 10) trước vì ảnh hưởng đến mAP, sau đó fix cache (Cell 5 & 7) cho tiện dùng. 🚀


# ─── Dataset helper ───────────────────────────────────────────────────────── class ShopeeDataset(Dataset): def __init__(self, df, img_dir, transform=None): self.df, self.img_dir, self.transform = df, img_dir, transform def __len__(self): return len(self.df) def __getitem__

pasted

sửa giúp tui luôn đ bạn

12:45 PM

Orchestrated code correction for feature caching implementation
Orchestrated code correction for feature caching implementation
Thay đổi so với code gốc
Mình thêm 1 helper function _load_or_extract_clip() để tái sử dụng logic cache cho cả 3 tập:

python
# Code gốc — lặp lại logic 3 lần, không cache val/test ❌
if os.path.exists(GAL_IMG_PATH):
    gallery_img_feat = np.load(...)
else:
    gallery_img_feat = extract_clip_image_features(...)
    np.save(...)

# val/test không có cache → chạy lại mỗi lần restart
val_img_feat  = extract_clip_image_features(df_val, IMG_DIR)
test_img_feat = extract_clip_image_features(df_test, IMG_DIR)
python
# Code mới — 1 helper function, cả 3 tập đều có cache ✅
gallery_img_feat, gallery_txt_feat = _load_or_extract_clip(
    GAL_IMG_PATH, GAL_TXT_PATH, df_gallery, IMG_DIR, 'gallery')

val_img_feat, val_txt_feat = _load_or_extract_clip(
    VAL_IMG_PATH, VAL_TXT_PATH, df_val, IMG_DIR, 'val')

test_img_feat, test_txt_feat = _load_or_extract_clip(
    TEST_IMG_PATH, TEST_TXT_PATH, df_test, IMG_DIR, 'test')
Lần đầu chạy: extract + lưu file .npy vào /content/features/ Lần sau restart: load thẳng từ file → tiết kiệm ~20-30 phút. 🚀


Cell5 fixed
PY 

You've used 90% of your session limit
Upgrade




Claude is AI and can make mistakes. Please double-check responses.
Cell5 fixed · PY
# ─── Dataset helper ─────────────────────────────────────────────────────────
class ShopeeDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df, self.img_dir, self.transform = df, img_dir, transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, self.df.iloc[idx]['image'])
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (256, 256), (128,128,128))
        if self.transform:
            return self.transform(img)
        return img
 
 
def l2_norm(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, 1e-12)
 
 
@torch.no_grad()
def extract_clip_image_features(df_in, img_dir, batch_size=BATCH_SIZE_CLIP):
    all_feats = []
    if USE_MOBILECLIP:
        ds     = ShopeeDataset(df_in, img_dir, transform=clip_preprocess)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        for imgs in tqdm(loader, desc='🖼️ CLIP image'):
            feats = clip_model.encode_image(imgs.to(DEVICE))
            all_feats.append(feats.cpu().float().numpy())
    else:
        ds = ShopeeDataset(df_in, img_dir)
        for i in tqdm(range(0, len(ds), batch_size), desc='🖼️ CLIP image'):
            batch  = [ds[j] for j in range(i, min(i+batch_size, len(ds)))]
            inputs = clip_preprocess(images=batch, return_tensors='pt',
                                     padding=True).to(DEVICE)
            feats  = clip_model.get_image_features(**inputs)
            all_feats.append(feats.cpu().float().numpy())
    return l2_norm(np.vstack(all_feats))
 
 
@torch.no_grad()
def extract_clip_text_features(df_in, batch_size=256):
    titles, all_feats = df_in['title'].fillna('').tolist(), []
    if USE_MOBILECLIP:
        for i in tqdm(range(0, len(titles), batch_size), desc='📝 CLIP text'):
            tokens = clip_tokenizer(titles[i:i+batch_size]).to(DEVICE)
            feats  = clip_model.encode_text(tokens)
            all_feats.append(feats.cpu().float().numpy())
    else:
        for i in tqdm(range(0, len(titles), batch_size), desc='📝 CLIP text'):
            inputs = clip_preprocess(
                text=titles[i:i+batch_size], return_tensors='pt',
                padding=True, truncation=True, max_length=77
            ).to(DEVICE)
            feats = clip_model.get_text_features(**inputs)
            all_feats.append(feats.cpu().float().numpy())
    return l2_norm(np.vstack(all_feats))
 
 
# ─── Helper: load hoặc extract + save ───────────────────────────────────────
def _load_or_extract_clip(img_path, txt_path, df_in, img_dir, label):
    """Load từ cache nếu có, không thì extract rồi lưu lại."""
    img_ok = os.path.exists(img_path) and np.load(img_path).shape[0] == len(df_in)
    txt_ok = os.path.exists(txt_path) and np.load(txt_path).shape[0] == len(df_in)
 
    if img_ok and txt_ok:
        print(f'✅ Load cached {label} features')
        return np.load(img_path), np.load(txt_path)
 
    print(f'⚙️  Extracting {label} image features...')
    img_feat = extract_clip_image_features(df_in, img_dir)
    np.save(img_path, img_feat)
 
    print(f'⚙️  Extracting {label} text features...')
    txt_feat = extract_clip_text_features(df_in)
    np.save(txt_path, txt_feat)
 
    print(f'💾 Saved {label} → {img_path}')
    return img_feat, txt_feat
 
 
# ─── Cache paths ─────────────────────────────────────────────────────────────
GAL_IMG_PATH  = os.path.join(FEAT_DIR, 'gallery_clip_image.npy')
GAL_TXT_PATH  = os.path.join(FEAT_DIR, 'gallery_clip_text.npy')
VAL_IMG_PATH  = os.path.join(FEAT_DIR, 'val_clip_image.npy')
VAL_TXT_PATH  = os.path.join(FEAT_DIR, 'val_clip_text.npy')
TEST_IMG_PATH = os.path.join(FEAT_DIR, 'test_clip_image.npy')
TEST_TXT_PATH = os.path.join(FEAT_DIR, 'test_clip_text.npy')
 
# ─── Extract / Load ──────────────────────────────────────────────────────────
gallery_img_feat, gallery_txt_feat = _load_or_extract_clip(
    GAL_IMG_PATH, GAL_TXT_PATH, df_gallery, IMG_DIR, 'gallery')
 
val_img_feat, val_txt_feat = _load_or_extract_clip(
    VAL_IMG_PATH, VAL_TXT_PATH, df_val, IMG_DIR, 'val')
 
test_img_feat, test_txt_feat = _load_or_extract_clip(
    TEST_IMG_PATH, TEST_TXT_PATH, df_test, IMG_DIR, 'test')
 
# ─── Verify ──────────────────────────────────────────────────────────────────
print(f'\n✅ gallery img/txt : {gallery_img_feat.shape} / {gallery_txt_feat.shape}')
print(f'✅ val     img/txt : {val_img_feat.shape} / {val_txt_feat.shape}')
print(f'✅ test    img/txt : {test_img_feat.shape} / {test_txt_feat.shape}')
 


## ✂️ Cell 6: YOLO Crop — Toàn bộ Gallery

In [ ]:
import math
from ultralytics import YOLO

# ─── Utility helpers (từ two_stage_pipeline.py) ─────────────────────────────
def _clamp_box(x1, y1, x2, y2, W, H):
    x1, y1 = max(0.0, x1), max(0.0, y1)
    x2, y2 = min(float(W), x2), min(float(H), y2)
    return (x1, y1, x2, y2) if (x2 > x1 and y2 > y1) else None

def _choose_best_box(boxes, scores, W, H):
    img_area = max(W * H, 1)
    cx_img, cy_img = W / 2.0, H / 2.0
    diag = math.hypot(W, H)
    best_score, best_box = -1e9, None
    for box, conf in zip(boxes, scores):
        b = _clamp_box(*box[:4], W, H)
        if b is None: continue
        x1, y1, x2, y2 = b
        ar = (x2-x1)*(y2-y1) / img_area
        if ar < 0.01 or ar > 0.95: continue
        cx, cy = (x1+x2)/2, (y1+y2)/2
        dist   = math.hypot(cx-cx_img, cy-cy_img)
        center = 1.0 - dist / max(diag, 1.0)
        area   = min(ar / 0.45, 1.0)
        score  = float(conf) + 0.20*center + 0.15*area
        if score > best_score:
            best_score, best_box = score, b
    return best_box

def _center_crop(img: Image.Image, ratio=FALLBACK_RATIO) -> Image.Image:
    w, h = img.size
    nw, nh = int(w*ratio), int(h*ratio)
    return img.crop(((w-nw)//2, (h-nh)//2, (w-nw)//2+nw, (h-nh)//2+nh))


# ─── Load YOLO ──────────────────────────────────────────────────────────────
print(f'⏳ Loading YOLO: {YOLO_WEIGHTS}')
yolo_model = YOLO(YOLO_WEIGHTS)   # downloads yolo11n.pt automatically if needed
print('✅ YOLO loaded')


# ─── Crop gallery ───────────────────────────────────────────────────────────
def crop_gallery(df_in, img_dir, save_dir, conf=YOLO_CONF, pad=CROP_PADDING):
    os.makedirs(save_dir, exist_ok=True)
    stats = {'yolo': 0, 'fallback': 0, 'error': 0}

    for _, row in tqdm(df_in.iterrows(), total=len(df_in), desc='YOLO crop'):
        fname    = row['image']
        src_path = os.path.join(img_dir, fname)
        dst_path = os.path.join(save_dir, fname)

        if os.path.exists(dst_path):   # skip if already cropped
            continue
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)

        try:
            pil = Image.open(src_path).convert('RGB')
            W, H = pil.size
            det  = yolo_model.predict(src_path, imgsz=YOLO_IMGSZ,
                                      conf=conf, verbose=False)[0]
            boxes  = det.boxes.xyxy.cpu().numpy() if det.boxes else np.zeros((0,4))
            confs  = det.boxes.conf.cpu().numpy() if det.boxes else np.zeros(0)
            best   = _choose_best_box(boxes, confs, W, H) if len(boxes) else None

            if best is not None:
                x1, y1, x2, y2 = best
                bw, bh = x2-x1, y2-y1
                x1 = max(0, x1 - bw*pad); y1 = max(0, y1 - bh*pad)
                x2 = min(W, x2 + bw*pad); y2 = min(H, y2 + bh*pad)
                cropped = pil.crop((x1, y1, x2, y2))
                stats['yolo'] += 1
            else:
                cropped = _center_crop(pil)
                stats['fallback'] += 1

        except Exception:
            try:
                pil = Image.open(src_path).convert('RGB')
                cropped = _center_crop(pil)
            except Exception:
                cropped = Image.new('RGB', (224, 224), 128)
            stats['error'] += 1

        cropped.save(dst_path)

    total = len(df_in)
    print(f'\n✅ Crop done | yolo={stats["yolo"]} ({stats["yolo"]/total:.1%}) | '
          f'fallback={stats["fallback"]} | error={stats["error"]}')
    return stats


crop_stats = crop_gallery(df_gallery, IMG_DIR, CROP_DIR)

# Kiểm tra số file crop
n_cropped = sum(1 for f in df_gallery['image'] if os.path.exists(os.path.join(CROP_DIR, f)))
print(f'📁 Crop files: {n_cropped:,} / {len(df_gallery):,}')

## 🦕 Cell 7: Trích xuất DINOv2 Features (trên YOLO crops)

In [ ]:
# ─── Giải phóng VRAM từ MobileCLIP trước khi load DINOv2 ───────────────────
# (T4 chỉ có 15 GB, không thể giữ cả hai model cùng lúc)
del clip_model
gc.collect()
torch.cuda.empty_cache()
print(f'🗑️  Freed CLIP. VRAM trống: '
      f'{torch.cuda.memory_reserved(0)/1e9:.1f} GB reserved')

# ─── DINOv2 transform ───────────────────────────────────────────────────────
DINO_TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])


class CropDataset(Dataset):
    def __init__(self, df, crop_dir, transform):
        self.paths = [
            os.path.join(crop_dir, row['image'])
            for _, row in df.iterrows()
        ]
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), 128)
        return self.transform(img)


@torch.no_grad()
def extract_dino_features(df_in, crop_dir, dino_model, batch_size=BATCH_SIZE_DINO):
    ds     = CropDataset(df_in, crop_dir, DINO_TRANSFORM)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    all_feats = []
    for batch in tqdm(loader, desc='🦕 DINOv2 extract'):
        batch = batch.to(DEVICE, non_blocking=True)
        out   = dino_model.forward_features(batch)
        feats = out['x_norm_clstoken'] if isinstance(out, dict) else out
        if feats.ndim == 3: feats = feats[:, 0, :]
        feats = F.normalize(feats, dim=-1)
        all_feats.append(feats.cpu().float().numpy())
    return l2_norm(np.vstack(all_feats))


# ─── Helper: load hoặc extract + save ───────────────────────────────────────
def _load_or_extract_dino(cache_path, df_in, crop_dir, dino_model, label):
    """Load từ cache nếu có, không thì extract rồi lưu lại."""
    if os.path.exists(cache_path) and np.load(cache_path).shape[0] == len(df_in):
        print(f'✅ Load cached {label} DINOv2 features')
        return np.load(cache_path)

    print(f'⚙️  Extracting {label} DINOv2 features...')
    feat = extract_dino_features(df_in, crop_dir, dino_model)
    np.save(cache_path, feat)
    print(f'💾 Saved {label} → {cache_path}')
    return feat


# ─── Load DINOv2 ────────────────────────────────────────────────────────────
print(f'⏳ Loading {DINO_MODEL} from torch.hub...')
dino_model = torch.hub.load('facebookresearch/dinov2', DINO_MODEL, verbose=False)
dino_model = dino_model.eval().to(DEVICE)
print('✅ DINOv2 loaded')

# ─── Cache paths ─────────────────────────────────────────────────────────────
GAL_DINO_PATH  = os.path.join(FEAT_DIR, f'gallery_dino_crop_{DINO_MODEL}.npy')
VAL_DINO_PATH  = os.path.join(FEAT_DIR, f'val_dino_crop_{DINO_MODEL}.npy')
TEST_DINO_PATH = os.path.join(FEAT_DIR, f'test_dino_crop_{DINO_MODEL}.npy')

# ─── Extract / Load ──────────────────────────────────────────────────────────
gallery_dino_feat = _load_or_extract_dino(
    GAL_DINO_PATH,  df_gallery, CROP_DIR, dino_model, 'gallery')

val_dino_feat = _load_or_extract_dino(
    VAL_DINO_PATH,  df_val,     CROP_DIR, dino_model, 'val')

test_dino_feat = _load_or_extract_dino(
    TEST_DINO_PATH, df_test,    CROP_DIR, dino_model, 'test')

# ─── Verify ──────────────────────────────────────────────────────────────────
print(f'\n✅ gallery_dino_feat : {gallery_dino_feat.shape}')
print(f'✅ val_dino_feat     : {val_dino_feat.shape}')
print(f'✅ test_dino_feat    : {test_dino_feat.shape}')

# ─── Giải phóng DINOv2, reload MobileCLIP cho bước tuning ──────────────────
del dino_model
gc.collect()
torch.cuda.empty_cache()
print('\n🗑️  DINOv2 unloaded. Reload MobileCLIP cho grid search...')

## 🔄 Cell 7b: Reload MobileCLIP sau khi extract DINOv2

In [ ]:
# Reload MobileCLIP (cần cho fuse_and_normalize và evaluate)
if USE_MOBILECLIP:
    import mobileclip
    clip_model, _, clip_preprocess = mobileclip.create_model_and_transforms(
        MOBILECLIP_VARIANT, pretrained=MOBILECLIP_CKPT
    )
    clip_tokenizer = mobileclip.get_tokenizer(MOBILECLIP_VARIANT)
    clip_model = clip_model.to(DEVICE).eval()
else:
    from transformers import CLIPModel, CLIPProcessor
    clip_model      = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE).eval()
    clip_preprocess = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
    clip_tokenizer  = None

print(f'✅ {MODEL_LABEL} reloaded')

# ─── Core utility functions ─────────────────────────────────────────────────
def fuse_clip(img_feats, txt_feats, alpha):
    """Weighted fusion + L2 normalize."""
    fused = alpha * img_feats + (1 - alpha) * txt_feats
    return l2_norm(fused)


def build_faiss_index(features, use_gpu=True):
    idx = faiss.IndexFlatIP(features.shape[1])
    if use_gpu and faiss.get_num_gpus() > 0:
        res = faiss.StandardGpuResources()
        idx = faiss.index_cpu_to_gpu(res, 0, idx)
    idx.add(np.ascontiguousarray(features.astype(np.float32)))
    return idx


def score_fusion(dino_s, clip_s, beta, top_k=FINAL_K):
    """Min-max normalize both, then blend."""
    def _mm(a): return np.clip((a - a.min())/(a.max()-a.min()+1e-8), 0, 1)
    fused = beta * _mm(dino_s.astype(np.float32)) + \
            (1-beta) * _mm(clip_s.astype(np.float32))
    k = min(top_k, len(fused))
    part  = np.argpartition(-fused, k-1)[:k]
    order = np.argsort(-fused[part])
    return part[order], fused[part[order]]


def get_gt_dict(df_in):
    gt = {}
    for _, grp in df_in.groupby('label_group'):
        ids = set(grp['posting_id'].tolist())
        for pid in ids: gt[pid] = ids
    return gt


print('✅ Utility functions ready')

## 🔍 Cell 8: Grid Search Alpha — Stage 1 (MobileCLIP Only)

In [ ]:
import matplotlib.pyplot as plt

def evaluate_stage1(q_img, q_txt, g_img, g_txt, query_df, gallery_df, alpha, K=5):
    """Evaluate MobileCLIP-only (Stage 1)."""
    q_fused = fuse_clip(q_img, q_txt, alpha)
    g_fused = fuse_clip(g_img, g_txt, alpha)
    index   = build_faiss_index(g_fused)
    _, idxs = index.search(np.ascontiguousarray(q_fused), K+1)
    gt      = get_gt_dict(gallery_df)
    g_pids  = gallery_df['posting_id'].tolist()

    ap_list, p1_list, r5_list = [], [], []
    for i, row in enumerate(query_df.itertuples()):
        relevant = gt.get(row.posting_id, set()) - {row.posting_id}
        if not relevant: continue
        retrieved = [g_pids[j] for j in idxs[i] if g_pids[j] != row.posting_id][:K]
        hits, ap = 0, 0.0
        for rank, pid in enumerate(retrieved, 1):
            if pid in relevant: hits += 1; ap += hits/rank
        ap_list.append(ap / min(len(relevant), K))
        p1_list.append(1.0 if retrieved and retrieved[0] in relevant else 0.0)
        r5_list.append(len(set(retrieved) & relevant) / len(relevant))
    return {'mAP@5': np.mean(ap_list), 'Precision@1': np.mean(p1_list), 'Recall@5': np.mean(r5_list)}


print('🔍 Grid Search Alpha (Stage 1 only)...')
alpha_results = []
best_alpha, best_map_alpha = 0.5, 0.0

for alpha in np.arange(0.0, 1.05, 0.1):
    alpha = round(float(alpha), 1)
    m = evaluate_stage1(val_img_feat, val_txt_feat,
                        gallery_img_feat, gallery_txt_feat,
                        df_val, df_gallery, alpha)
    alpha_results.append({'alpha': alpha, **m})
    marker = ' ← BEST' if m['mAP@5'] > best_map_alpha else ''
    print(f'  alpha={alpha:.1f} | mAP@5={m["mAP@5"]:.4f} | '
          f'P@1={m["Precision@1"]:.4f} | R@5={m["Recall@5"]:.4f}{marker}')
    if m['mAP@5'] > best_map_alpha:
        best_map_alpha, best_alpha = m['mAP@5'], alpha

alpha_df = pd.DataFrame(alpha_results)
print(f'\n🏆 Best alpha = {best_alpha:.1f}  (val mAP@5 = {best_map_alpha:.4f})')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alpha_df['alpha'], alpha_df['mAP@5'], 'o-', color='#3B82F6', lw=2, label='mAP@5')
ax.axvline(best_alpha, ls='--', color='red', alpha=0.6, label=f'Best α={best_alpha}')
ax.set_xlabel('Alpha', fontsize=12); ax.set_ylabel('mAP@5', fontsize=12)
ax.set_title('Grid Search: Alpha (Stage 1 — MobileCLIP only)', fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'grid_alpha.png'), dpi=150)
plt.show()

## 🔍 Cell 9: Grid Search Retrieval-K (Stage 2, β=1.0)

In [ ]:
def evaluate_two_stage(q_img, q_txt, q_dino,
                        g_img, g_txt, g_dino,
                        query_df, gallery_df,
                        alpha, retrieval_k, beta, K=FINAL_K):
    """Full 2-stage evaluation on pre-computed features."""
    # Stage 1: FAISS
    q_fused = fuse_clip(q_img, q_txt, alpha)
    g_fused = fuse_clip(g_img, g_txt, alpha)
    index   = build_faiss_index(g_fused)

    gt     = get_gt_dict(gallery_df)
    g_pids = gallery_df['posting_id'].tolist()
    g_lbls = gallery_df['label_group'].values

    ap_list, p1_list, r5_list = [], [], []

    for i in tqdm(range(len(query_df)), desc=f'Eval α={alpha} K={retrieval_k} β={beta}', leave=False):
        qid      = query_df.iloc[i]['posting_id']
        relevant = gt.get(qid, set()) - {qid}
        if not relevant: continue

        # FAISS search
        q_vec = np.ascontiguousarray(q_fused[i:i+1])
        dists, idxs = index.search(q_vec, retrieval_k + 1)
        cand_idxs   = idxs[0][idxs[0] >= 0]
        clip_scores = dists[0][idxs[0] >= 0]

        self_mask   = np.array([g_pids[j] != qid for j in cand_idxs])
        cand_idxs   = cand_idxs[self_mask]
        clip_scores = clip_scores[self_mask]

        if len(cand_idxs) == 0:
            ap_list.append(0.0); p1_list.append(0.0); r5_list.append(0.0)
            continue

        # Stage 2: DINOv2 rerank
        dino_scores = g_dino[cand_idxs] @ q_dino[i]   # cosine (features L2-normed)
        local_top, _ = score_fusion(dino_scores, clip_scores, beta=beta, top_k=K)
        global_top   = cand_idxs[local_top]
        retrieved    = [g_pids[j] for j in global_top]

        # Metrics
        hits, ap = 0, 0.0
        for rank, pid in enumerate(retrieved[:K], 1):
            if pid in relevant: hits += 1; ap += hits/rank
        ap_list.append(ap / min(len(relevant), K))
        p1_list.append(1.0 if retrieved and retrieved[0] in relevant else 0.0)
        r5_list.append(len(set(retrieved[:K]) & relevant) / len(relevant))

    return {
        'mAP@5'      : float(np.mean(ap_list)),
        'Precision@1': float(np.mean(p1_list)),
        'Recall@5'   : float(np.mean(r5_list)),
    }


# ─── Grid K (DINOv2 only, beta=1.0 để isolate effect của K) ─────────────────
print('🔍 Grid Search Retrieval-K (β=1.0, DINOv2 only)...')
k_results = []
best_k, best_map_k = 100, 0.0

for k in [50, 100, 150, 200]:
    m = evaluate_two_stage(
        val_img_feat, val_txt_feat, val_dino_feat,
        gallery_img_feat, gallery_txt_feat, gallery_dino_feat,
        df_val, df_gallery,
        alpha=best_alpha, retrieval_k=k, beta=1.0
    )
    k_results.append({'retrieval_k': k, **m})
    marker = ' ← BEST' if m['mAP@5'] > best_map_k else ''
    print(f'  K={k:3d} | mAP@5={m["mAP@5"]:.4f} | '
          f'P@1={m["Precision@1"]:.4f} | R@5={m["Recall@5"]:.4f}{marker}')
    if m['mAP@5'] > best_map_k:
        best_map_k, best_k = m['mAP@5'], k

k_df = pd.DataFrame(k_results)
print(f'\n🏆 Best K = {best_k}  (val mAP@5 = {best_map_k:.4f})')

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(r['retrieval_k']) for r in k_results],
       [r['mAP@5'] for r in k_results], color='#10B981')
ax.set_xlabel('Retrieval K'); ax.set_ylabel('mAP@5')
ax.set_title('Grid Search: Retrieval-K (β=1.0)', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'grid_k.png'), dpi=150)
plt.show()

## 🔍 Cell 10: Grid Search Beta — Full Pipeline

In [ ]:
print('🔍 Grid Search Beta (full pipeline)...')
beta_results = []
best_beta, best_map_beta = 0.3, 0.0

for beta in np.arange(0.0, 1.05, 0.1):
    beta = round(float(beta), 1)
    m = evaluate_two_stage(
        val_img_feat, val_txt_feat, val_dino_feat,
        gallery_img_feat, gallery_txt_feat, gallery_dino_feat,
        df_val, df_gallery,
        alpha=best_alpha, retrieval_k=best_k, beta=beta
    )
    beta_results.append({'beta': beta, **m})
    marker = ' ← BEST' if m['mAP@5'] > best_map_beta else ''
    print(f'  beta={beta:.1f} | mAP@5={m["mAP@5"]:.4f} | '
          f'P@1={m["Precision@1"]:.4f} | R@5={m["Recall@5"]:.4f}{marker}')
    if m['mAP@5'] > best_map_beta:
        best_map_beta, best_beta = m['mAP@5'], beta

beta_df = pd.DataFrame(beta_results)
print(f'\n🏆 Best beta = {best_beta:.1f}  (val mAP@5 = {best_map_beta:.4f})')

# ─── Summary hyperparameters ────────────────────────────────────────────────
best_params = {
    'alpha'      : float(best_alpha),
    'retrieval_k': int(best_k),
    'beta'       : float(best_beta),
    'val_mAP@5'  : float(best_map_beta),
}
hp_path = os.path.join(RESULTS_DIR, 'best_hyperparams.json')
with open(hp_path, 'w') as f:
    json.dump(best_params, f, indent=2)
print(f'\n💾 Hyperparameters saved → {hp_path}')
print(json.dumps(best_params, indent=2))

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(beta_df['beta'], beta_df['mAP@5'], 's-', color='#8B5CF6', lw=2)
ax.plot(beta_df['beta'], beta_df['Precision@1'], '^--', color='#F59E0B', lw=1.5, label='P@1')
ax.plot(beta_df['beta'], beta_df['Recall@5'],    'v--', color='#10B981', lw=1.5, label='R@5')
ax.axvline(best_beta, ls='--', color='red', alpha=0.7, label=f'Best β={best_beta}')
ax.set_xlabel('Beta (DINOv2 weight)', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Grid Search: Beta — Full 2-Stage Pipeline', fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'grid_beta.png'), dpi=150)
plt.show()

## 🧪 Cell 11: Đánh giá Test Set
> ⚠️ **QUAN TRỌNG**: Chỉ chạy cell này **1 lần duy nhất** sau khi đã chốt hyperparameters!

In [ ]:
print('=' * 60)
print('🧪 FINAL TEST SET EVALUATION (chạy 1 lần!)')
print('=' * 60)
print(f'   alpha       = {best_alpha}')
print(f'   retrieval_k = {best_k}')
print(f'   beta        = {best_beta}')
print()

test_metrics = evaluate_two_stage(
    test_img_feat, test_txt_feat, test_dino_feat,
    gallery_img_feat, gallery_txt_feat, gallery_dino_feat,
    df_test, df_gallery,
    alpha=best_alpha, retrieval_k=best_k, beta=best_beta
)

print('\n📊 FINAL RESULTS (Test Set):')
print(f'   mAP@5        = {test_metrics["mAP@5"]:.4f}')
print(f'   Precision@1  = {test_metrics["Precision@1"]:.4f}')
print(f'   Recall@5     = {test_metrics["Recall@5"]:.4f}')

# Save
results_payload = {
    'model'      : f'{MODEL_LABEL} + DINOv2-Crop ({DINO_MODEL})',
    'yolo'       : YOLO_WEIGHTS,
    'hyperparams': best_params,
    'test_metrics': test_metrics,
    'crop_stats' : crop_stats,
}
out_path = os.path.join(RESULTS_DIR, 'final_results_twostage.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results_payload, f, indent=2, ensure_ascii=False)
print(f'\n💾 Results saved → {out_path}')

## 📊 Cell 12: So sánh với Baseline & Bảng Báo cáo

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

baseline_rows = [
    {'Phương pháp': 'Baseline TF-IDF',
     'mAP@5': 0.5204, 'Precision@1': 0.5922, 'Recall@5': 0.6349},
    {'Phương pháp': 'Baseline ResNet50',
     'mAP@5': 0.5268, 'Precision@1': 0.6328, 'Recall@5': 0.5293},
    {'Phương pháp': 'Baseline MobileCLIP (Stage 1)',
     'mAP@5': 0.7708, 'Precision@1': 0.7937, 'Recall@5': 0.7430},
    {'Phương pháp': f'Tuần 4 — 2-Stage: {MODEL_LABEL} + YOLO-DINOv2',
     'mAP@5'      : test_metrics['mAP@5'],
     'Precision@1': test_metrics['Precision@1'],
     'Recall@5'   : test_metrics['Recall@5']},
]

report_df = pd.DataFrame(baseline_rows)
metric_cols = ['mAP@5', 'Precision@1', 'Recall@5']
report_df[metric_cols] = report_df[metric_cols].round(4)

# Lưu CSV
report_csv = os.path.join(RESULTS_DIR, 'REPORT_twostage_comparison.csv')
report_df.to_csv(report_csv, index=False, encoding='utf-8-sig')

print('📊 Bảng so sánh:')
display(report_df.style
    .highlight_max(subset=metric_cols, color='#86EFAC')
    .format({c: '{:.4f}' for c in metric_cols}))

# Tính delta so với MobileCLIP baseline
base = report_df[report_df['Phương pháp'].str.contains('Stage 1')].iloc[0]
new  = report_df.iloc[-1]
print('\n🎯 Cải thiện so với MobileCLIP Baseline:')
for col in metric_cols:
    delta = new[col] - base[col]
    sign  = '+' if delta >= 0 else ''
    print(f'   {col:15s}: {base[col]:.4f} → {new[col]:.4f}  ({sign}{delta:.4f} = {sign}{delta*100:.2f}%)')

# ─── Bar chart ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#94A3B8', '#94A3B8', '#3B82F6', '#10B981']

for ax, metric in zip(axes, metric_cols):
    vals   = report_df[metric].tolist()
    labels = [p.split('—')[-1].strip() if '—' in p else p
              for p in report_df['Phương pháp']]
    bars = ax.bar(range(len(vals)), vals, color=colors, edgecolor='white', width=0.6)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=8)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('So sánh các phương pháp — Shopee Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'comparison_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'\n💾 Saved → {report_csv}')

## 🖼️ Cell 13: Visualize Kết quả Truy vấn Top-5

In [ ]:
def visualize_query(query_idx, df_query, df_gallery,
                     q_img_feat, q_txt_feat, q_dino_feat,
                     g_img_feat, g_txt_feat, g_dino_feat,
                     img_dir, crop_dir,
                     alpha, retrieval_k, beta, K=5, show_crop=True):
    """Hiển thị query và top-5 retrieved ảnh kèm nhãn TP/FP."""

    row   = df_query.iloc[query_idx]
    q_lbl = row['label_group']

    # Run pipeline
    q_fused = fuse_clip(q_img_feat[query_idx:query_idx+1],
                         q_txt_feat[query_idx:query_idx+1], alpha)
    g_fused = fuse_clip(g_img_feat, g_txt_feat, alpha)
    index   = build_faiss_index(g_fused)
    dists, idxs = index.search(np.ascontiguousarray(q_fused), retrieval_k+1)

    cand     = idxs[0][idxs[0] >= 0]
    c_dists  = dists[0][idxs[0] >= 0]
    d_scores = g_dino_feat[cand] @ q_dino_feat[query_idx]
    local_top, scores = score_fusion(d_scores, c_dists, beta=beta, top_k=K)
    top_global = cand[local_top]

    # Plot
    fig, axes = plt.subplots(2 if show_crop else 1, K+1,
                             figsize=(3*(K+1), 3*(2 if show_crop else 1)+1))
    if not show_crop: axes = [axes]

    def _load(path):
        try: return Image.open(path).convert('RGB')
        except: return Image.new('RGB', (224, 224), 200)

    for row_i, (ax_row, use_crop) in enumerate(zip(axes, [False, True] if show_crop else [False])):
        d = crop_dir if use_crop else img_dir
        label = '(crop)' if use_crop else '(orig)'

        q_img = _load(os.path.join(d, row['image']))
        ax_row[0].imshow(q_img)
        ax_row[0].set_title(f'QUERY {label}\n{str(q_lbl)[:12]}', fontsize=8, color='navy')
        ax_row[0].axis('off')

        for rank, (g_idx, sc) in enumerate(zip(top_global, scores), 1):
            g_row  = df_gallery.iloc[g_idx]
            g_img  = _load(os.path.join(d, g_row['image']))
            is_tp  = g_row['label_group'] == q_lbl
            color  = '#16A34A' if is_tp else '#DC2626'
            tag    = '✅ TP' if is_tp else '❌ FP'
            ax_row[rank].imshow(g_img)
            ax_row[rank].set_title(f'Rank {rank} {tag}\n{sc:.3f}', fontsize=8, color=color)
            ax_row[rank].axis('off')
            for spine in ax_row[rank].spines.values():
                spine.set_edgecolor(color); spine.set_linewidth(3)

    title = (f'Query #{query_idx} | α={alpha} K={retrieval_k} β={beta} | '
             f'Title: {row["title"][:50]}')
    fig.suptitle(title, fontsize=9)
    plt.tight_layout()
    plt.show()


# ─── Visualize 3 queries ngẫu nhiên ─────────────────────────────────────────
rng = np.random.default_rng(42)
sample_idxs = rng.choice(len(df_test), size=3, replace=False)

for q_idx in sample_idxs:
    visualize_query(
        int(q_idx), df_test, df_gallery,
        test_img_feat, test_txt_feat, test_dino_feat,
        gallery_img_feat, gallery_txt_feat, gallery_dino_feat,
        IMG_DIR, CROP_DIR,
        alpha=best_alpha, retrieval_k=best_k, beta=best_beta,
        show_crop=True
    )

## 🔍 Cell 14: Demo Single Query (Production-style)

In [ ]:
# ─── Demo: Truy vấn 1 ảnh bất kỳ ────────────────────────────────────────────
DEMO_IDX   = 0                        # Thay đổi index này để xem query khác
DEMO_ROW   = df_test.iloc[DEMO_IDX]
DEMO_PATH  = os.path.join(IMG_DIR, DEMO_ROW['image'])
DEMO_TITLE = DEMO_ROW['title']

print(f'🔍 Demo Query:')
print(f'   Image : {DEMO_PATH}')
print(f'   Title : {DEMO_TITLE}')
print(f'   Label : {DEMO_ROW["label_group"]}')

t0 = time.perf_counter()

# --- Stage 1: MobileCLIP ---
q_img_vec = test_img_feat[DEMO_IDX:DEMO_IDX+1]
q_txt_vec = test_txt_feat[DEMO_IDX:DEMO_IDX+1]
q_fused   = fuse_clip(q_img_vec, q_txt_vec, best_alpha)
g_fused   = fuse_clip(gallery_img_feat, gallery_txt_feat, best_alpha)
idx       = build_faiss_index(g_fused)
t_faiss   = time.perf_counter()
dists, idxs = idx.search(np.ascontiguousarray(q_fused), best_k + 1)
t_faiss   = time.perf_counter() - t_faiss

cand      = idxs[0][idxs[0] >= 0]
c_dists   = dists[0][idxs[0] >= 0]

# --- Stage 2: DINOv2 ---
t_dino   = time.perf_counter()
d_scores = gallery_dino_feat[cand] @ test_dino_feat[DEMO_IDX]
local_top, final_scores = score_fusion(d_scores, c_dists, best_beta, top_k=5)
top5     = cand[local_top]
t_dino   = time.perf_counter() - t_dino
t_total  = time.perf_counter() - t0

print(f'\n⏱️  Latency breakdown:')
print(f'   FAISS search : {t_faiss*1000:.1f} ms')
print(f'   DINOv2 rerank: {t_dino*1000:.1f} ms')
print(f'   Total        : {t_total*1000:.1f} ms')

print(f'\n🏆 Top-5 Results:')
for rank, (g_idx, sc) in enumerate(zip(top5, final_scores), 1):
    g_row  = df_gallery.iloc[g_idx]
    is_tp  = g_row['label_group'] == DEMO_ROW['label_group']
    tag    = '✅ TP' if is_tp else '❌ FP'
    print(f'   Rank {rank} {tag} | score={sc:.4f} | {g_row["title"][:60]}')

# Show images
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
def _load(p):
    try: return Image.open(p).convert('RGB')
    except: return Image.new('RGB', (224,224), 200)

axes[0].imshow(_load(DEMO_PATH))
axes[0].set_title(f'QUERY\n{DEMO_TITLE[:20]}', fontsize=8, color='navy')
axes[0].axis('off')

for rank, (g_idx, sc) in enumerate(zip(top5, final_scores), 1):
    g_row = df_gallery.iloc[g_idx]
    gpath = os.path.join(IMG_DIR, g_row['image'])
    is_tp = g_row['label_group'] == DEMO_ROW['label_group']
    c     = '#16A34A' if is_tp else '#DC2626'
    axes[rank].imshow(_load(gpath))
    axes[rank].set_title(f'Rank {rank}\n{"✅ TP" if is_tp else "❌ FP"} {sc:.3f}',
                         fontsize=8, color=c)
    axes[rank].axis('off')
    for sp in axes[rank].spines.values(): sp.set_edgecolor(c); sp.set_linewidth(3)

plt.suptitle(f'Demo Query: "{DEMO_TITLE[:70]}"', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'demo_query.png'), dpi=150)
plt.show()

---
## ✅ Pipeline hoàn tất!

### Files đã tạo trong `/content/results/`:
| File | Nội dung |
|------|----------|
| `best_hyperparams.json` | α, K, β tối ưu |
| `final_results_twostage.json` | Metrics trên test set |
| `REPORT_twostage_comparison.csv` | Bảng so sánh với baselines |
| `grid_alpha.png` | Biểu đồ grid search α |
| `grid_k.png` | Biểu đồ grid search K |
| `grid_beta.png` | Biểu đồ grid search β |
| `comparison_chart.png` | Bar chart so sánh tất cả methods |
| `demo_query.png` | Ảnh demo truy vấn |

### Files features trong `/content/features/`:
| File | Shape |
|------|-------|
| `gallery_clip_image.npy` | (34250, 512) |
| `gallery_clip_text.npy` | (34250, 512) |
| `gallery_dino_crop_dinov2_vits14.npy` | (34250, 384) |

> 💡 **Tip**: Copy `/content/features/` và `/content/results/` lên Google Drive để không mất khi runtime reset!